# MP3 to WAV

In [ ]:
!pip install pydub
!pip install ffmpeg

  Preparing metadata (setup.py) ... done
  Created wheel for ffmpeg: filename=ffmpeg-1.4-py3-none-any.whl size=6083 sha256=45c2a4db2769d5d5b28d33df4ee6595083a1641a4895d068e15a3ad9868f2087
  Stored in directory: /root/.cache/pip/wheels/26/21/0c/c26e09dff860a9071683e279445262346e008a9a1d2142c4ad
Successfully built ffmpeg


In [ ]:
from google.colab import files

import zipfile
from pathlib import Path

from pydub import AudioSegment

import shutil

In [ ]:
uploaded = files.upload()

Saving bible.zip to bible (1).zip


In [ ]:
zip_path = Path('bible.zip')
input_dir = Path('bible')

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('.')

output_dir = Path('wav_bible')
output_dir.mkdir(exist_ok=True)

for input_file in input_dir.glob('*.mp3'):
    output_file = output_dir / f'{input_file.stem}.wav'

    sound = AudioSegment.from_mp3(input_file)

    sound = sound.set_frame_rate(16000)
    sound = sound.set_channels(1)
    sound = sound.set_sample_width(2)

    sound.export(output_file, format='wav')

    print(f'Converted: {input_file.name} -> {output_file.name}')

Converted: Luke_018.mp3 -> Luke_018.wav
Converted: CHKBS 053 «Эмтэма Чиниткин аʼӄанмыкэн уттынпын...».mp3 -> CHKBS 053 «Эмтэма Чиниткин аʼӄанмыкэн уттынпын...».wav
Converted: CHKBS 028 Тиркэрмэӈэв Есфир.mp3 -> CHKBS 028 Тиркэрмэӈэв Есфир.wav
Converted: CHKGP 000 Вэймэну ԓынъёйгыт каԓевэтгавыԓьэгыт!.mp3 -> CHKGP 000 Вэймэну ԓынъёйгыт каԓевэтгавыԓьэгыт!.wav
Converted: CKTPO_0_1.mp3 -> CKTPO_0_1.wav
Converted: CHKBS 026 Давид Израилыԓьэн тиркэрым.mp3 -> CHKBS 026 Давид Израилыԓьэн тиркэрым.wav
Converted: Luke_013.mp3 -> Luke_013.wav
Converted: CHKBS 032 Коргыэнанымӈыԓятгыргын Иисус уʼрэтыԓьыԓӄыԓ.mp3 -> CHKBS 032 Коргыэнанымӈыԓятгыргын Иисус уʼрэтыԓьыԓӄыԓ.wav
Converted: CHKBS 041 Ӈайыткынкэнат нинъэйвыт – тэнчичӈыт Моисейын.mp3 -> CHKBS 041 Ӈайыткынкэнат нинъэйвыт – тэнчичӈыт Моисейын.wav
Converted: CHKBS 039 Мынгыткэн ӈиръэ пароԓ рыгъеватъёттэ Иисус-Христосына.mp3 -> CHKBS 039 Мынгыткэн ӈиръэ пароԓ рыгъеватъёттэ Иисус-Христосына.wav
Converted: CHKBS 000 Вэймэну ԓынъёйгыт каԓевэтгавыԓьэгыт

In [ ]:
total_ms = 0

for audio_file in output_dir.rglob('*.wav'):
    sound = AudioSegment.from_wav(audio_file)
    total_ms += len(sound)

total_seconds = int(total_ms / 1000)

hours = total_seconds // 3600
minutes = (total_seconds % 3600) // 60
seconds = total_seconds % 60

print(f'Total duration: {hours:02d}:{minutes:02d}:{seconds:02d}')

Total duration: 07:29:24


In [ ]:
shutil.make_archive('bible_wav', 'zip', 'wav_bible')

'/content/bible_wav.zip'

## Primary pause segmentation and text matching

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import xml.etree.ElementTree as ET

from pydub import AudioSegment
from pydub.silence import detect_nonsilent

from google.colab import files
import zipfile

In [ ]:
uploaded = files.upload()

zip_path = Path(next(iter(uploaded.keys())))

Saving wav_bible.zip to wav_bible (1).zip
Saving texts_bible.zip to texts_bible (1).zip


NameError: name 'zipfile' is not defined

In [ ]:
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('.')

In [ ]:
audio_dir = Path('wav_bible')
text_dir = Path('texts_bible')

tier_id = 'transcription'

In [ ]:
def normalize_text(text):
    return ' '.join(text.split())


def find_segments_by_pauses(
    audio,
    min_len_ms=5000,
    max_len_ms=15000,
    min_silence_len=700,
    keep_silence_ms=250
):
    silence_thresh = audio.dBFS - 14

    nonsilent_ranges = detect_nonsilent(
        audio,
        min_silence_len=min_silence_len,
        silence_thresh=silence_thresh,
        seek_step=10
    )

    segments = []

    for start, end in nonsilent_ranges:
        start = max(0, start - keep_silence_ms)
        end = min(len(audio), end + keep_silence_ms)
        segments.append([start, end])

    if not segments:
        return [[0, len(audio)]]

    merged = []
    cur_start, cur_end = segments[0]

    for start, end in segments[1:]:
        new_len = end - cur_start
        cur_len = cur_end - cur_start

        if new_len <= max_len_ms or cur_len < min_len_ms:
            cur_end = end
        else:
            merged.append([cur_start, cur_end])
            cur_start, cur_end = start, end

    merged.append([cur_start, cur_end])

    final_segments = []

    for start, end in merged:
        while end - start > max_len_ms:
            final_segments.append([start, start + max_len_ms])
            start += max_len_ms

        if end - start > 1000:
            final_segments.append([start, end])

    if len(final_segments) >= 2 and final_segments[-1][1] - final_segments[-1][0] < min_len_ms:
        final_segments[-2][1] = final_segments[-1][1]
        final_segments.pop()

    return final_segments

In [ ]:
def split_text_by_durations(text, durations_ms):
    text_parts = []

    if not durations_ms:
        return []

    if not text:
        return ['' for _ in durations_ms]

    total_duration = sum(durations_ms)

    if total_duration <= 0:
        return ['' for _ in durations_ms]

    cuts = []
    current_duration = 0
    previous_cut = 0

    for duration in durations_ms[:-1]:
        current_duration += duration
        target = round(len(text) * current_duration / total_duration)

        punctuation_positions = [
            i + 1
            for i, char in enumerate(text)
            if char in '.?!;:' and i > previous_cut
        ]

        nearby_punctuation = [
            pos
            for pos in punctuation_positions
            if abs(pos - target) <= 150
        ]

        if nearby_punctuation:
            cut = min(nearby_punctuation, key=lambda pos: abs(pos - target))
        else:
            space_positions = [
                i
                for i, char in enumerate(text)
                if char == ' ' and i > previous_cut
            ]

            if space_positions:
                cut = min(space_positions, key=lambda pos: abs(pos - target))
            else:
                cut = target

        cut = max(cut, previous_cut)
        cut = min(cut, len(text))

        cuts.append(cut)
        previous_cut = cut

    prev = 0

    for cut in cuts:
        text_parts.append(text[prev:cut].strip())
        prev = cut

    text_parts.append(text[prev:].strip())

    return text_parts

In [ ]:
def make_eaf(audio_file, output_eaf, segments, text_parts, tier_id='transcription'):
    now = datetime.now(timezone.utc).isoformat()

    root = ET.Element(
        'ANNOTATION_DOCUMENT',
        {
            'AUTHOR': '',
            'DATE': now,
            'FORMAT': '3.0',
            'VERSION': '3.0'
        }
    )

    header = ET.SubElement(
        root,
        'HEADER',
        {
            'MEDIA_FILE': '',
            'TIME_UNITS': 'milliseconds'
        }
    )

    ET.SubElement(
        header,
        'MEDIA_DESCRIPTOR',
        {
            'MEDIA_URL': audio_file.resolve().as_uri(),
            'MIME_TYPE': 'audio/x-wav',
            'RELATIVE_MEDIA_URL': f'./{audio_file.name}'
        }
    )

    time_order = ET.SubElement(root, 'TIME_ORDER')

    tier = ET.SubElement(
        root,
        'TIER',
        {
            'TIER_ID': tier_id,
            'LINGUISTIC_TYPE_REF': 'default-lt'
        }
    )

    for i, ((start, end), text_part) in enumerate(zip(segments, text_parts), start=1):
        ts_start = f'ts{2 * i - 1}'
        ts_end = f'ts{2 * i}'

        ET.SubElement(
            time_order,
            'TIME_SLOT',
            {
                'TIME_SLOT_ID': ts_start,
                'TIME_VALUE': str(int(start))
            }
        )

        ET.SubElement(
            time_order,
            'TIME_SLOT',
            {
                'TIME_SLOT_ID': ts_end,
                'TIME_VALUE': str(int(end))
            }
        )

        annotation = ET.SubElement(tier, 'ANNOTATION')

        alignable = ET.SubElement(
            annotation,
            'ALIGNABLE_ANNOTATION',
            {
                'ANNOTATION_ID': f'a{i}',
                'TIME_SLOT_REF1': ts_start,
                'TIME_SLOT_REF2': ts_end
            }
        )

        value = ET.SubElement(alignable, 'ANNOTATION_VALUE')
        value.text = text_part

    ET.SubElement(
        root,
        'LINGUISTIC_TYPE',
        {
            'LINGUISTIC_TYPE_ID': 'default-lt',
            'TIME_ALIGNABLE': 'true'
        }
    )

    tree = ET.ElementTree(root)
    ET.indent(tree, space='  ', level=0)
    tree.write(output_eaf, encoding='UTF-8', xml_declaration=True)

In [ ]:
text_files = [
    path
    for path in text_dir.rglob('*')
    if path.suffix.lower() == '.txt'
]

text_by_stem = {
    path.stem: path
    for path in text_files
}

audio_files = [
    path
    for path in audio_dir.rglob('*')
    if path.suffix.lower() == '.wav'
]

processed = 0
skipped = 0

for audio_file in sorted(audio_files):
    text_file = text_by_stem.get(audio_file.stem)

    if text_file is None:
        print(f'Skipped, no text: {audio_file.name}')
        skipped += 1
        continue

    audio = AudioSegment.from_file(audio_file)

    text = text_file.read_text(encoding='utf-8-sig')
    text = normalize_text(text)

    segments = find_segments_by_pauses(audio)
    durations_ms = [end - start for start, end in segments]
    text_parts = split_text_by_durations(text, durations_ms)

    output_eaf = audio_file.with_suffix('.eaf')

    make_eaf(
        audio_file=audio_file,
        output_eaf=output_eaf,
        segments=segments,
        text_parts=text_parts,
        tier_id=tier_id
    )

    print(f'Created: {output_eaf.name}, segments: {len(segments)}')
    processed += 1

print()
print(f'Processed: {processed}')
print(f'Skipped without text: {skipped}')

Skipped, no text: CHKBS 000 Вэймэну ԓынъёйгыт каԓевэтгавыԓьэгыт!.wav
Skipped, no text: CHKBS 001 Ымнутэйиквикин энантомгаквыргын.wav
Skipped, no text: CHKBS 002 Энантомгаквыргын ымнутэйиквикин (йыԓгииԓ).wav
Skipped, no text: CHKBS 003 Рытомгаквыргын ыʼттъыёԓкэн оʼравэтԓьаргэн.wav
Skipped, no text: CHKBS 004 Адам ынкъам Ева гэнвыԓинэт таӈычьытванвэпы.wav
Skipped, no text: CHKBS 005 Эмԓятгыргын. Нойын ыʼтвиӈын.wav
Skipped, no text: CHKBS 006 Тиркыӄымчучьын ынкъам вэтгычьат Тэнантомгыӈэн оʼравэтԓьак рээн.wav
Skipped, no text: CHKBS 007 Авраам.wav
Skipped, no text: CHKBS 008 Тэнантомгыӈэн энатват Авраамына.wav
Skipped, no text: CHKBS 009 Авраамына Исаак энатрэԓтаароӈо ԓыӈыркынин.wav
Skipped, no text: CHKBS 010 Иосиф – Иаковын ыʼԓгыэкык.wav
Skipped, no text: CHKBS 011 Иосифын рэтыт.wav
Skipped, no text: CHKBS 012 Йичьэмиттумгэ Иосиф нынвиԓыткувӄин.wav
Skipped, no text: CHKBS 013 Иосиф – инэнԓеԓьын армарагынрэтыԓьык.wav
Skipped, no text: CHKBS 014 Иосиф восӄырак.wav
Skipped, no text: CHKBS 0

In [ ]:
zip_name = 'eaf_bible.zip'

with zipfile.ZipFile(zip_name, 'w') as zipf:
    for eaf_file in audio_dir.rglob('*.eaf'):
        zipf.write(eaf_file, arcname=eaf_file.name)

files.download(zip_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Splitting text and audio after .eaf correction

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

import xml.etree.ElementTree as ET
from pydub import AudioSegment

import shutil

In [ ]:
uploaded = files.upload()

zip_path = Path(next(iter(uploaded.keys())))

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('.')

Saving eaf_bible.zip to eaf_bible.zip


In [ ]:
audio_dir = Path('wav_bible')
eaf_dir = Path('eaf_bible')

tier_id = 'transcription'

output_dir = Path('segmented_dataset')
output_dir.mkdir(exist_ok=True)

In [ ]:
def get_time_slots(root):
    time_slots = {}

    for time_slot in root.findall('.//TIME_SLOT'):
        time_slot_id = time_slot.attrib['TIME_SLOT_ID']
        time_value = int(time_slot.attrib['TIME_VALUE'])
        time_slots[time_slot_id] = time_value

    return time_slots


def find_tier(root, tier_id):
    for tier in root.findall('.//TIER'):
        if tier.attrib.get('TIER_ID') == tier_id:
            return tier

    return None


def get_annotations_from_eaf(eaf_file, tier_id):
    tree = ET.parse(eaf_file)
    root = tree.getroot()

    time_slots = get_time_slots(root)
    tier = find_tier(root, tier_id)

    if tier is None:
        raise ValueError(f'Tier "{tier_id}" not found in {eaf_file.name}')

    annotations = []

    for annotation in tier.findall('.//ALIGNABLE_ANNOTATION'):
        start_slot = annotation.attrib['TIME_SLOT_REF1']
        end_slot = annotation.attrib['TIME_SLOT_REF2']

        start_ms = time_slots[start_slot]
        end_ms = time_slots[end_slot]

        value_element = annotation.find('ANNOTATION_VALUE')
        text = value_element.text if value_element is not None and value_element.text else ''
        text = ' '.join(text.split())

        annotations.append(
            {
                'start_ms': start_ms,
                'end_ms': end_ms,
                'text': text
            }
        )

    annotations = sorted(annotations, key=lambda item: item['start_ms'])

    return annotations


def export_segments(audio_file, eaf_file, tier_id, output_dir):
    audio = AudioSegment.from_file(audio_file)
    annotations = get_annotations_from_eaf(eaf_file, tier_id)

    chapter_output_dir = output_dir / audio_file.stem
    chapter_output_dir.mkdir(exist_ok=True)

    segment_number = 1
    skipped_empty = 0
    skipped_invalid = 0

    for annotation in annotations:
        start_ms = annotation['start_ms']
        end_ms = annotation['end_ms']
        text = annotation['text']

        if end_ms <= start_ms:
            skipped_invalid += 1
            continue

        if not text:
            skipped_empty += 1
            continue

        segment_name = f'{audio_file.stem}_segment-{segment_number}'

        segment_audio = audio[start_ms:end_ms]

        segment_audio.export(
            chapter_output_dir / f'{segment_name}.wav',
            format='wav'
        )

        (chapter_output_dir / f'{segment_name}.txt').write_text(
            text,
            encoding='utf-8'
        )

        segment_number += 1

    exported = segment_number - 1

    return {
        'exported': exported,
        'skipped_empty': skipped_empty,
        'skipped_invalid': skipped_invalid
    }

In [ ]:
eaf_files = [
    path
    for path in eaf_dir.rglob('*.eaf')
]

audio_files_by_stem = {
    path.stem: path
    for path in audio_dir.rglob('*.wav')
}

total_exported = 0
total_skipped_empty = 0
total_skipped_invalid = 0
skipped_no_audio = 0

for eaf_file in sorted(eaf_files):
    audio_file = audio_files_by_stem.get(eaf_file.stem)

    if audio_file is None:
        print(f'Skipped, no wav: {eaf_file.name}')
        skipped_no_audio += 1
        continue

    result = export_segments(
        audio_file=audio_file,
        eaf_file=eaf_file,
        tier_id=tier_id,
        output_dir=output_dir
    )

    total_exported += result['exported']
    total_skipped_empty += result['skipped_empty']
    total_skipped_invalid += result['skipped_invalid']

    print(
        f'{eaf_file.name}: '
        f'exported {result["exported"]}, '
        f'skipped empty {result["skipped_empty"]}, '
        f'skipped invalid {result["skipped_invalid"]}'
    )

print()
print(f'Total exported: {total_exported}')
print(f'Total skipped empty annotations: {total_skipped_empty}')
print(f'Total skipped invalid annotations: {total_skipped_invalid}')
print(f'Total skipped EAF without WAV: {skipped_no_audio}')

Jonah_002.eaf: exported 11, skipped empty 0, skipped invalid 0
Jonah_003.eaf: exported 11, skipped empty 0, skipped invalid 0
Jonah_004.eaf: exported 11, skipped empty 0, skipped invalid 0
Luke_001.eaf: exported 66, skipped empty 0, skipped invalid 0
Luke_002.eaf: exported 48, skipped empty 0, skipped invalid 0
Luke_003.eaf: exported 34, skipped empty 0, skipped invalid 0
Luke_004.eaf: exported 40, skipped empty 0, skipped invalid 0
Luke_005.eaf: exported 41, skipped empty 0, skipped invalid 0
Luke_006.eaf: exported 59, skipped empty 0, skipped invalid 0
Luke_007.eaf: exported 68, skipped empty 0, skipped invalid 0
Luke_008.eaf: exported 82, skipped empty 1, skipped invalid 0
Luke_009.eaf: exported 75, skipped empty 0, skipped invalid 0
Luke_010.eaf: exported 62, skipped empty 0, skipped invalid 0
Luke_011.eaf: exported 72, skipped empty 0, skipped invalid 0
Luke_012.eaf: exported 81, skipped empty 0, skipped invalid 0
Luke_013.eaf: exported 53, skipped empty 0, skipped invalid 0
Luke_

In [ ]:
shutil.make_archive('segmented_dataset', 'zip', output_dir)
files.download('segmented_dataset.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Merging train, test, dev spreadsheets from Safonova et al. 2022

In [ ]:
import pandas as pd

In [ ]:
train = pd.read_csv('train_new.tsv', sep='\t')
test = pd.read_csv('test_new.tsv', sep='\t')
dev = pd.read_csv('dev_new.tsv', sep='\t')

In [ ]:
data = pd.concat([train, test, dev])
data = data.sort_values(by=['path'])
data.head()

,path,sentence,duration
94,11_01-18.00_segment-10.wav,ымыԓьо кэлит тыӈивылӄылти ӄликкин январьтагнэты,3.300
852,11_01-18.00_segment-11.wav,миӈкыри кэлийвылӄылти рыкэтъоӈаннэн инэнлеԓьын...,8.968
156,11_01-18.00_segment-12.wav,ынан тывнэн нымытваԓьыт микынти гэнъэллинэт эн...,17.503
664,11_01-18.00_segment-13.wav,чама микынти эннукэ гитлинэт нэмыӄэй рыԓьататъ...,6.081
151,11_01-18.00_segment-14.wav,отчётак кэлийвылӄыл чама миӈкы ръаынныӈыттынвы...,6.933


In [ ]:
data.to_csv('data.tsv', sep="\t")

In [ ]:
data.duration.sum()

np.float64(11399.623004000001)

# Finalizing spreadsheet with all files

## Getting durations for radio_chuklang_metadata

In [ ]:
import pandas as pd

from pathlib import Path
import zipfile

import wave

In [ ]:
radio_chuklang_metadata = pd.read_csv('radio_chuklang_metadata.tsv', sep='\t')

In [ ]:
radio_chuklang_metadata.sample(6)

,resource,path,sentence,duration
497,radio,21_12-18.00_segment-19.wav,рыгйивевкы герескивлин ванкыттаменырак аномавк...,NaN
576,radio,22_12-18.00_segment-31.wav,гамӈылтэтлен инспектор въээкин гибдд татьяна п...,NaN
1530,chuklang,Raven and fox_13.wav,вэнлыгиыʼм ӄол нивӄин вэнлыги и нинивӄин,NaN
1115,chuklang,GUM_20.wav,нылейгым гыргочаткын гыргоча ынӄэн нинэгитэйгы...,NaN
1237,chuklang,Ice Age_9.wav,ынкы имлетыкъыма чит ӈутку мургынутэ ваԓьыт ка...,NaN
877,chuklang,Being a child_27.wav,аны евъев евъев,NaN


In [ ]:
zip_path = Path('chuklang_radio_wavs.zip')
extract_dir = Path('chuklang_radio_wavs')

extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [ ]:
def get_wav_duration(wav_path):
    with wave.open(str(wav_path), 'rb') as wav_file:
        frames = wav_file.getnframes()
        frame_rate = wav_file.getframerate()
        return frames / frame_rate

In [ ]:
durations = {}

for wav_file in extract_dir.rglob('*.wav'):
    parts = wav_file.relative_to(extract_dir).parts

    resource = None

    for part in parts:
        if part in ['radio', 'chuklang']:
            resource = part
            break

    if resource is None:
        print(f'Cannot infer resource for {wav_file}')
        continue

    key = (resource, wav_file.name)
    durations[key] = get_wav_duration(wav_file)

print(f'Found durations for {len(durations)} wav files')

Found durations for 1829 wav files


In [ ]:
def find_duration(row):
    key = (row['resource'], row['path'])
    return durations.get(key)

radio_chuklang_metadata['duration'] = radio_chuklang_metadata.apply(
    find_duration,
    axis=1
)

In [ ]:
missing = radio_chuklang_metadata[radio_chuklang_metadata['duration'].isna()]

In [ ]:
radio_chuklang_metadata.duration.describe()

,duration
count,1828.000000
mean,6.184243
std,4.092567
min,0.520000
25%,3.117750
50%,5.160000
75%,8.348299
max,29.513016


## Creating bible_metadata

In [ ]:
from pathlib import Path
import zipfile
import wave
import pandas as pd

from google.colab import files

In [ ]:
zip_path = Path('bible.zip')
extract_dir = Path('bible')

extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print('Files extracted:', len(list(extract_dir.iterdir())))

Files extracted: 2911


In [ ]:
resource_name = 'bible'

rows = []

wav_files = sorted(extract_dir.glob('*.wav'))

for wav_file in wav_files:
    txt_file = wav_file.with_suffix('.txt')

    if not txt_file.exists():
        print(f'Skipped, no txt: {wav_file.name}')
        continue

    sentence = txt_file.read_text(encoding='utf-8')
    sentence = ' '.join(sentence.split())

    duration = get_wav_duration(wav_file)

    rows.append(
        {
            'resource': resource_name,
            'path': wav_file.name,
            'sentence': sentence,
            'duration': duration
        }
    )

bible_df = pd.DataFrame(rows)

print('Rows:', len(bible_df))
bible_df.head()

Rows: 1455


,resource,path,sentence,duration
0,bible,Jonah_001_segment-1.wav,кэԓикэԓ ионан,1.896961
1,bible,Jonah_001_segment-10.wav,"Миӈкы гынин нутэнут, ръавараткэнайгыт?» Иквъи ...",5.900000
2,bible,Jonah_001_segment-11.wav,"«Гым – еврейыԓьэгым, тыԓымаԓявыркын Этынвэты Т...",16.369977
3,bible,Jonah_001_segment-12.wav,«Гэрэӄигыт ынӈин гитигыт»? Ӄэԓюӄ-ым нэԓкыԓгъэн...,14.850023
4,bible,Jonah_001_segment-13.wav,«Миӈкри мынынтыгыт ынратвэнво морыкы аӈӄы? Ӄэԓ...,10.320000


In [ ]:
print(bible_df.duration.sum())
bible_df.duration.describe()

13579.714916099772


,duration
count,1455.000000
mean,9.333137
std,4.415456
min,0.810000
25%,6.075000
50%,9.010000
75%,12.280000
max,28.863000


## Creating final metadata table

In [ ]:
metadata = pd.concat([radio_chuklang_metadata, bible_df])

In [ ]:
metadata = metadata.reset_index(drop=True)
metadata.head()

,resource,path,sentence,duration
0,radio,11_01-18.00_segment-10.wav,ымыԓьо кэлит тыӈивылӄылти ӄликкин январьтагнэты,3.300000
1,radio,11_01-18.00_segment-11.wav,миӈкыри кэлийвылӄылти рыкэтъоӈаннэн инэнлеԓьын...,8.967982
2,radio,11_01-18.00_segment-12.wav,ынан тывнэн нымытваԓьыт микынти гэнъэллинэт эн...,17.503129
3,radio,11_01-18.00_segment-13.wav,чама микынти эннукэ гитлинэт нэмыӄэй рыԓьататъ...,6.080998
4,radio,11_01-18.00_segment-14.wav,отчётак кэлийвылӄыл чама миӈкы ръаынныӈыттынвы...,6.933016


In [ ]:
summary = (
    metadata
    .groupby('resource', as_index=False)
    .agg(files_count=('path', 'count'), total_duration_sec=('duration', 'sum'))
)

def seconds_to_hms(seconds):
    seconds = int(round(seconds))
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f'{hours:02d}:{minutes:02d}:{secs:02d}'


summary['total_duration_hms'] = summary['total_duration_sec'].apply(seconds_to_hms)

summary

,resource,files_count,total_duration_sec,total_duration_hms
0,bible,1455,13579.714916,03:46:20
1,chuklang,999,4446.691005,01:14:07
2,radio,829,6858.104580,01:54:18


## Texts clearance

In [ ]:
import pandas as pd

from collections import Counter
import unicodedata

import regex as re

from google.colab import files

In [ ]:
metadata = pd.read_csv('metadata.tsv', sep='\t')

In [ ]:
metadata.loc[metadata['path'] == 'Water cart_2.wav', 'sentence'] = 'ковлёрвоор зил сто пятьдесят седьмой ковлёрвыткоԓьын танӈынотайпы етыԓьын'

In [ ]:
radio_mask = metadata['resource'] == 'radio'

metadata.loc[radio_mask, 'sentence'] = (
    metadata.loc[radio_mask, 'sentence']
    .astype(str)
    .str.replace('–', ' ', regex=False)
    .str.replace('\u0301', '', regex=False)  # ́
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

In [ ]:
quote_chars = set('„“”"\'‘’«»‚‛')

def clean_bible_keep_quotes(text):
    if pd.isna(text):
        return ''

    text = str(text)

    cleaned_chars = []

    for char in text:
        category = unicodedata.category(char)

        # Цифры заменяем на пробел
        if category.startswith('N'):
            cleaned_chars.append(' ')

        # Пунктуацию заменяем на пробел, кроме кавычек из quote_chars
        elif category.startswith('P') and char not in quote_chars:
            cleaned_chars.append(' ')

        else:
            cleaned_chars.append(char)

    text = ''.join(cleaned_chars)

    # Убираем лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def keep_only_word_internal_quotes(text):
    if pd.isna(text):
        return ''

    text = str(text)
    chars = list(text)
    cleaned = []

    for i, char in enumerate(chars):
        if char in quote_chars:
            prev_char = chars[i - 1] if i > 0 else ''
            next_char = chars[i + 1] if i + 1 < len(chars) else ''

            if prev_char.isalpha() and next_char.isalpha():
                cleaned.append(char)
            else:
                cleaned.append(' ')
        else:
            cleaned.append(char)

    text = ''.join(cleaned)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
bible_mask = metadata['resource'] == 'bible'

metadata.loc[bible_mask, 'sentence'] = (metadata.loc[bible_mask, 'sentence'].apply(clean_bible_keep_quotes))
metadata.loc[bible_mask, 'sentence'] = (metadata.loc[bible_mask, 'sentence'].apply(keep_only_word_internal_quotes))

In [ ]:
bible_text = ' '.join(
    metadata.loc[metadata['resource'] == 'bible', 'sentence']
    .dropna()
    .astype(str)
)

char_counts = Counter(bible_text)

symbols = []

for char, count in char_counts.items():
    if not char.isalpha() and not char.isspace():
        symbols.append(
            {
                'char': char,
                'count': count
            }
        )

symbols_df = (
    pd.DataFrame(symbols)
    .sort_values('count', ascending=False)
)

symbols_df

,char,count
0,’,85


In [ ]:
target_char = '’'

rows_with_char = metadata[
    (metadata['resource'] == 'bible') &
    (metadata['sentence'].astype(str).str.contains(target_char, regex=False, na=False))
]

print('Rows:', len(rows_with_char))

rows_with_char[['path', 'sentence']].head(20)

Rows: 83


,path,sentence
1891,Luke_001_segment-2.wav,о’равэтԓьат микырык ынӄэн моотагнэпы гэԓьуԓин ...
1910,Luke_001_segment-37.wav,ытԓён етгъи захариян ярагты ынкъам иквъи а’мын...
1957,Luke_002_segment-2.wav,ы’ттъыёԓ ынӈот нарыԓгыӈӈонат оʼравэтԓьат титэ ...
1998,Luke_003_segment-13.wav,а’ԓыгаттэ энмэч варкын уттыкинмык ӄача уттуут ...
2053,Luke_004_segment-32.wav,ымыԓьо гэԓгиничгытэтԓинэт а’мын ым вай како ко...
2080,Luke_005_segment-20.wav,о’птытъар оʼравэтԓьата нэрэтын энанрыёԓгыткыны...
2087,Luke_005_segment-27.wav,гым тытэгъеӈыркын иӈӄун торгынан ԓыги нъыԓгытк...
2124,Luke_006_segment-23.wav,кычьымыԓьыторэ тэргыԓьыторэ тури ракоргытвэӈыт...
2127,Luke_006_segment-26.wav,э’ткиӈыԓьытури аʼнӄавытваԓьыторэ игыр тури раг...
2128,Luke_006_segment-27.wav,э’ткиӈыԓьытури ымыԓьорык анъяйвыёторэ ӄэԓюӄ ым...


In [ ]:
metadata['sentence'] = (metadata['sentence'].fillna('').astype(str).str.lower())

In [ ]:
STANDARD_APOSTROPHE = "'"

APOSTROPHE_LIKE = ''.join([
    chr(0x02BC),  # ʼ MODIFIER LETTER APOSTROPHE
    chr(0x2019),  # ’ RIGHT SINGLE QUOTATION MARK
    chr(0x2018),  # ‘ LEFT SINGLE QUOTATION MARK
    chr(0x0027),  # ' APOSTROPHE
    chr(0x02BB),  # ʻ MODIFIER LETTER TURNED COMMA
    chr(0x0294),  # ʔ LATIN LETTER GLOTTAL STOP
])


def replace_latin_inside_cyrillic_text(text):
    replacements = {
        'x': 'х',
        'c': 'с',
        'ž': 'ж',
    }

    for latin_char, cyrillic_char in replacements.items():
        pattern = (
            rf'(?<=\p{{Cyrillic}}){re.escape(latin_char)}'
            rf'|'
            rf'{re.escape(latin_char)}(?=\p{{Cyrillic}})'
        )

        text = re.sub(pattern, cyrillic_char, text)

    return text


def clean_asr_text(text):
    if pd.isna(text):
        return ''

    text = str(text)
    text = unicodedata.normalize('NFC', text)
    text = text.lower()

    # Убираем combining acute accent: ́
    text = text.replace('\u0301', '')

    # Исправляем латинские символы внутри/рядом с кириллицей
    text = replace_latin_inside_cyrillic_text(text)

    # Внутри кириллического слова все апострофоподобные символы → обычный '
    text = re.sub(
        rf'(?<=\p{{Cyrillic}})[{re.escape(APOSTROPHE_LIKE)}](?=\p{{Cyrillic}})',
        STANDARD_APOSTROPHE,
        text
    )

    # Апострофоподобные символы НЕ между кириллическими буквами убираем
    text = re.sub(
        rf'(?<!\p{{Cyrillic}})[{re.escape(APOSTROPHE_LIKE)}]',
        ' ',
        text
    )

    text = re.sub(
        rf'[{re.escape(APOSTROPHE_LIKE)}](?!\p{{Cyrillic}})',
        ' ',
        text
    )

    # Оставляем только кириллицу, пробелы и обычный '
    text = re.sub(
        r"[^'\p{Cyrillic}\s]+",
        ' ',
        text
    )

    # Схлопываем пробелы
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
metadata = metadata.copy()

metadata['sentence'] = metadata['sentence'].apply(clean_asr_text)

In [ ]:
for char in ['ʼ', '’', '‘', "'", 'ʻ', 'ʔ', 'x', 'c', 'ž', '\u0301']:
    count = (
        metadata['sentence']
        .dropna()
        .astype(str)
        .str.count(re.escape(char))
        .sum()
    )

    print(
        repr(char),
        f'U+{ord(char):04X}',
        unicodedata.name(char, 'UNKNOWN'),
        count
    )

'ʼ' U+02BC MODIFIER LETTER APOSTROPHE 0
'’' U+2019 RIGHT SINGLE QUOTATION MARK 0
'‘' U+2018 LEFT SINGLE QUOTATION MARK 0
"'" U+0027 APOSTROPHE 1003
'ʻ' U+02BB MODIFIER LETTER TURNED COMMA 0
'ʔ' U+0294 LATIN LETTER GLOTTAL STOP 0
'x' U+0078 LATIN SMALL LETTER X 0
'c' U+0063 LATIN SMALL LETTER C 0
'ž' U+017E LATIN SMALL LETTER Z WITH CARON 0
'́' U+0301 COMBINING ACUTE ACCENT 0


In [ ]:
all_text = ' '.join(metadata['sentence'].dropna().astype(str))
vocab = sorted(set(all_text))

for char in vocab:
    print(
        repr(char),
        f'U+{ord(char):04X}',
        unicodedata.name(char, 'UNKNOWN')
    )

' ' U+0020 SPACE
"'" U+0027 APOSTROPHE
'а' U+0430 CYRILLIC SMALL LETTER A
'б' U+0431 CYRILLIC SMALL LETTER BE
'в' U+0432 CYRILLIC SMALL LETTER VE
'г' U+0433 CYRILLIC SMALL LETTER GHE
'д' U+0434 CYRILLIC SMALL LETTER DE
'е' U+0435 CYRILLIC SMALL LETTER IE
'ж' U+0436 CYRILLIC SMALL LETTER ZHE
'з' U+0437 CYRILLIC SMALL LETTER ZE
'и' U+0438 CYRILLIC SMALL LETTER I
'й' U+0439 CYRILLIC SMALL LETTER SHORT I
'к' U+043A CYRILLIC SMALL LETTER KA
'л' U+043B CYRILLIC SMALL LETTER EL
'м' U+043C CYRILLIC SMALL LETTER EM
'н' U+043D CYRILLIC SMALL LETTER EN
'о' U+043E CYRILLIC SMALL LETTER O
'п' U+043F CYRILLIC SMALL LETTER PE
'р' U+0440 CYRILLIC SMALL LETTER ER
'с' U+0441 CYRILLIC SMALL LETTER ES
'т' U+0442 CYRILLIC SMALL LETTER TE
'у' U+0443 CYRILLIC SMALL LETTER U
'ф' U+0444 CYRILLIC SMALL LETTER EF
'х' U+0445 CYRILLIC SMALL LETTER HA
'ц' U+0446 CYRILLIC SMALL LETTER TSE
'ч' U+0447 CYRILLIC SMALL LETTER CHE
'ш' U+0448 CYRILLIC SMALL LETTER SHA
'щ' U+0449 CYRILLIC SMALL LETTER SHCHA
'ъ' U+044A CYRIL

In [ ]:
metadata.sample(5)

,resource,path,sentence,duration
47,radio,11_12-18.00_segment-25.wav,ненри кыткин танкалейпаты нырон йиԓьгык ынетоԓ...,9.370023
404,radio,19_01-18.00_segment-2.wav,еттык кыпалёмтеԓьгыткы торпыныԓьте гыым энаным...,5.492018
2968,bible,Luke_020_segment-15.wav,ытԓён вай миӈкри митгъэк мынӈивыгъэн гымнин ы'...,7.870000
1801,chuklang,Worms_1.wav,кымгытчитгъи кымъылгыӄэй,2.221995
3208,bible,Luke_023_segment-78.wav,суббота ы'ԓёк ытри гонтымытваԓенат этгиԓивкэ м...,7.190000


## Uploading

In [ ]:
metadata.to_csv('metadata.tsv', sep='\t', index=False, encoding='utf-8')

In [ ]:
files.download('metadata.tsv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Train-dev-test split

In [ ]:
!pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sklearn.model_selection import train_test_split
from google.colab import files

In [ ]:
metadata = pd.read_csv('metadata.tsv', sep='\t')

metadata = metadata.copy()
metadata = metadata.reset_index(drop=True)

metadata['duration'] = pd.to_numeric(metadata['duration'], errors='coerce')

metadata.head()

,resource,path,sentence,duration
0,radio,11_01-18.00_segment-10.wav,ымыԓьо кэлит тыӈивылӄылти ӄликкин январьтагнэты,3.300000
1,radio,11_01-18.00_segment-11.wav,миӈкыри кэлийвылӄылти рыкэтъоӈаннэн инэнлеԓьын...,8.967982
2,radio,11_01-18.00_segment-12.wav,ынан тывнэн нымытваԓьыт микынти гэнъэллинэт эн...,17.503129
3,radio,11_01-18.00_segment-13.wav,чама микынти эннукэ гитлинэт нэмыӄэй рыԓьататъ...,6.080998
4,radio,11_01-18.00_segment-14.wav,отчётак кэлийвылӄыл чама миӈкы ръаынныӈыттынвы...,6.933016


In [ ]:
def infer_source_id(row):
    resource = row['resource']
    stem = Path(str(row['path'])).stem

    # Jonah_001_segment-3 -> Jonah_001
    if resource in ['bible', 'radio']:
        return re.sub(r'_segment-\d+$', '', stem)

    # Not eating larvae_7 -> Not eating larvae
    if resource == 'chuklang':
        return re.sub(r'_\d+$', '', stem)

    raise ValueError(f'Unknown resource: {resource}')

metadata['source_id'] = metadata.apply(infer_source_id, axis=1)

metadata.sample(5)

,resource,path,sentence,duration,source_id
3183,bible,Luke_023_segment-55.wav,маравӄԓявыԓтэ нэмыӄэй нытвэтчатваӄэнат ачгыта ...,11.67,Luke_023
655,radio,25_12-18.00_segment-19.wav,кыликкин нырокавык геньгыенлин минкемиԓь гатай...,5.96,25_12-18.00
1282,chuklang,Kettle_32.wav,рэмкэ ивнин,2.02,Kettle
3071,bible,Luke_022_segment-3.wav,иуда эԓвэмиԓ а'йӈавъё искариот итыԓьын ӄол мын...,11.70,Luke_022
2189,bible,Luke_007_segment-29.wav,вай ым кычьымэты ваԓьын ынӄэн мэӈин ӄырым нымэ...,5.40,Luke_007


In [ ]:
def add_duration_bins(df, n_bins=3):
    df = df.copy()

    bins = pd.qcut(df['duration'], q=n_bins, labels=False, duplicates='drop')

    df['duration_bin'] = bins.astype(str)

    return df


parts = []

for resource, part in metadata.groupby('resource', sort=False):
    part = add_duration_bins(part)
    part['resource'] = resource
    parts.append(part)

metadata = pd.concat(parts, axis=0)
metadata = metadata.sort_index()

metadata.sample(5)

,resource,path,sentence,duration,source_id,duration_bin
468,radio,21_01-18.00_segment-20.wav,гетеньгыенлинет сенка тестайпы ниреккыликкин к...,6.885034,21_01-18.00,1
1870,bible,Jonah_004_segment-10.wav,ытԓён иквъи тыгтакавкэтгъак рыпэт тырэвъиӈыркы...,19.259000,Jonah_004,2
2972,bible,Luke_020_segment-19.wav,иисус ым тэӈгитэк ытри пынԓёгъэ ръэнут ым тывы...,7.160000,Luke_020,1
3142,bible,Luke_023_segment-18.wav,иродына гамаравӄԓявыԓма тармачьыӈэты ԓыгинъээ'...,14.380000,Luke_023,2
2313,bible,Luke_008_segment-8.wav,энанрэтԓявма чымӄык эрэтгъэт ръэтык ӄача ынкъа...,8.060000,Luke_008,1


In [ ]:
pd.crosstab(metadata['resource'], metadata['duration_bin'])

duration_bin,0,1,2
resource,,,
bible,485,486,484
chuklang,334,332,333
radio,277,276,276


In [ ]:
def split_resource(df, resource, test_size, dev_size=0.0, random_state=42):
    part = df[df['resource'] == resource].copy()

    stratify = part['duration_bin']

    train_dev_idx, test_idx = train_test_split(part.index, test_size=test_size,
        random_state=random_state, stratify=stratify)

    if dev_size == 0:
        df.loc[train_dev_idx, 'split'] = 'train'
        df.loc[test_idx, 'split'] = 'test'
        return df

    train_dev_part = df.loc[train_dev_idx].copy()
    stratify_train_dev = train_dev_part['duration_bin']

    relative_dev_size = dev_size / (1 - test_size)

    train_idx, dev_idx = train_test_split(train_dev_part.index, test_size=relative_dev_size,
        random_state=random_state, stratify=stratify_train_dev)

    df.loc[train_idx, 'split'] = 'train'
    df.loc[dev_idx, 'split'] = 'dev'
    df.loc[test_idx, 'split'] = 'test'

    return df

In [ ]:
RANDOM_STATE = 42

metadata['split'] = None

metadata = split_resource(metadata, resource='bible', test_size=0.10,
    dev_size=0.0, random_state=RANDOM_STATE)

metadata = split_resource(metadata, resource='radio', test_size=0.10,
    dev_size=0.0, random_state=RANDOM_STATE)

metadata = split_resource(metadata, resource='chuklang', test_size=0.20,
    dev_size=0.14, random_state=RANDOM_STATE)

In [ ]:
def seconds_to_hms(seconds):
    seconds = int(round(seconds))
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f'{hours:02d}:{minutes:02d}:{secs:02d}'

In [ ]:
split_summary = (
    metadata
    .groupby(['resource', 'split'], as_index=False)
    .agg(
        files_count=('path', 'count'),
        total_duration_sec=('duration', 'sum'),
        mean_duration=('duration', 'mean'),
        median_duration=('duration', 'median'),
        min_duration=('duration', 'min'),
        max_duration=('duration', 'max'),
        source_count=('source_id', 'nunique')
    )
)

split_summary['total_duration_hms'] = (split_summary['total_duration_sec'].apply(seconds_to_hms))

split_summary = split_summary[['resource', 'split', 'files_count',
        'total_duration_hms', 'source_count', 'mean_duration', 'median_duration',
        'min_duration', 'max_duration']]

split_summary

,resource,split,files_count,total_duration_hms,source_count,mean_duration,median_duration,min_duration,max_duration
0,bible,test,146,00:22:38,28,9.299658,8.767000,1.380000,19.710000
1,bible,train,1309,03:23:42,28,9.336872,9.040000,0.810000,28.863000
2,chuklang,dev,140,00:10:12,51,4.370514,3.535000,0.936984,17.711995
3,chuklang,test,200,00:14:13,58,4.267330,3.505000,0.650000,15.825000
4,chuklang,train,659,00:49:41,65,4.524056,3.601995,0.520000,24.420000
5,radio,test,83,00:11:22,26,8.222151,6.885034,1.420998,21.696009
6,radio,train,746,01:42:56,27,8.278373,7.492517,0.980998,29.513016


In [ ]:
source_split_summary = (
    metadata
    .groupby(['resource', 'split', 'source_id'], as_index=False)
    .agg(
        files_count=('path', 'count'),
        total_duration_sec=('duration', 'sum')
    )
)

total_by_resource_split = (
    source_split_summary
    .groupby(['resource', 'split'])['total_duration_sec']
    .transform('sum')
)

source_split_summary['share'] = (
    source_split_summary['total_duration_sec'] /
    total_by_resource_split
)

top_sources = (
    source_split_summary
    .sort_values(
        ['resource', 'split', 'share'],
        ascending=[True, True, False]
    )
    .groupby(['resource', 'split'])
    .head(5)
)

top_sources['total_duration_hms'] = (
    top_sources['total_duration_sec']
    .apply(seconds_to_hms)
)

top_sources[
    [
        'resource',
        'split',
        'source_id',
        'files_count',
        'total_duration_hms',
        'share'
    ]
]

,resource,split,source_id,files_count,total_duration_hms,share
15,bible,test,Luke_012,13,00:01:49,0.080619
14,bible,test,Luke_011,10,00:01:42,0.075382
8,bible,test,Luke_005,9,00:01:37,0.071138
16,bible,test,Luke_013,7,00:01:17,0.056380
17,bible,test,Luke_014,7,00:01:06,0.048453
39,bible,train,Luke_008,73,00:11:47,0.057828
32,bible,train,Luke_001,62,00:11:45,0.057703
40,bible,train,Luke_009,69,00:11:33,0.056684
53,bible,train,Luke_022,79,00:11:15,0.055241
43,bible,train,Luke_012,68,00:10:38,0.052167


In [ ]:
pd.crosstab(
    [metadata['resource'], metadata['source_id']],
    metadata['split']
)

split                 dev  test  train
resource source_id                    
bible    Jonah_001      0     3     16
         Jonah_002      0     1     10
         Jonah_003      0     2      9
         Jonah_004      0     1     10
         Luke_001       0     4     62
...                   ...   ...    ...
radio    26_01-18.00    0     3     31
         27_01-18.00    0     2     30
         28_01-18.00    0     4     28
         28_12-18.00    0     2     26
         30_12-18.00    0     3     27

[120 rows x 3 columns]

In [ ]:
metadata.sample(5)

,resource,path,sentence,duration,source_id,duration_bin,split
1793,chuklang,Water cart_9.wav,ӄорагынрэтыԓьыт вэчьыма н'ъетынэтэ мынъумэкэнм...,5.710000,Water cart,2,test
2395,bible,Luke_010_segment-11.wav,ынкъам эвыр тури еттык ръамайӈынымнымэты ынкъа...,16.320000,Luke_010,2,train
2503,bible,Luke_011_segment-53.wav,ынӈатаԓ уйӈэ эчимгъукыԓьытури эты ынӄэнына мик...,11.410000,Luke_011,2,train
907,chuklang,Boots_8.wav,ынкъамэ трилгытэквъэъым травэръэпыгъа трачапок...,9.428000,Boots,2,train
566,radio,22_12-18.00_segment-2.wav,еттык ӄыпалёмтэлгыткы торпыӈлытэ,3.428027,22_12-18.00,0,train


In [ ]:
metadata.to_csv('metadata_with_split.tsv', sep='\t', index=False, encoding='utf-8')

In [ ]:
files.download('metadata_with_split.tsv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Making HuggingFace Dataset
see https://huggingface.co/docs/datasets/v2.4.0/audio_load?utm_source=chatgpt.com

In [ ]:
!pip install -U datasets[audio] huggingface_hub pandas

In [ ]:
from pathlib import Path
import zipfile

from huggingface_hub import login, create_repo, HfApi

import pandas as pd

from datasets import Dataset, DatasetDict, Audio, Features, Value

from google.colab import userdata


In [ ]:
zip_path = Path('ckt_data.zip')
audio_root = Path('ckt_data')

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('.')

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [ ]:
metadata_path = Path('metadata_with_split.tsv')
audio_root = Path('audio')

metadata = pd.read_csv(metadata_path, sep='\t')

metadata.head()

,resource,path,sentence,duration,source_id,duration_bin,split
0,radio,11_01-18.00_segment-10.wav,ымыԓьо кэлит тыӈивылӄылти ӄликкин январьтагнэты,3.300000,11_01-18.00,0,train
1,radio,11_01-18.00_segment-11.wav,миӈкыри кэлийвылӄылти рыкэтъоӈаннэн инэнлеԓьын...,8.967982,11_01-18.00,1,train
2,radio,11_01-18.00_segment-12.wav,ынан тывнэн нымытваԓьыт микынти гэнъэллинэт эн...,17.503129,11_01-18.00,2,train
3,radio,11_01-18.00_segment-13.wav,чама микынти эннукэ гитлинэт нэмыӄэй рыԓьататъ...,6.080998,11_01-18.00,1,train
4,radio,11_01-18.00_segment-14.wav,отчётак кэлийвылӄыл чама миӈкы ръаынныӈыттынвы...,6.933016,11_01-18.00,1,train


In [ ]:
metadata['audio'] = metadata.apply(lambda row: str(audio_root / row['resource'] / row['path']), axis=1)

In [ ]:
missing_audio = metadata[~metadata['audio'].apply(lambda path: Path(path).exists())]

print('Missing audio files:', len(missing_audio))

missing_audio[['resource', 'path', 'audio']].head(20)

Missing audio files: 0


,resource,path,audio


In [ ]:
metadata.groupby(['resource', 'split']).size()

resource  split
bible     test      146
          train    1309
chuklang  dev       140
          test      200
          train     659
radio     test       83
          train     746
dtype: int64

In [ ]:
metadata_hf = metadata.copy()

metadata_hf['audio'] = metadata_hf['audio'].apply(lambda path: {'path': str(path), 'bytes': None})
metadata_hf['sentence'] = metadata_hf['sentence'].fillna('').astype(str)
metadata_hf['resource'] = metadata_hf['resource'].fillna('').astype(str)
metadata_hf['path'] = metadata_hf['path'].fillna('').astype(str)
metadata_hf['source_id'] = metadata_hf['source_id'].fillna('').astype(str)
metadata_hf['duration_bin'] = metadata_hf['duration_bin'].fillna('').astype(str)
metadata_hf['split'] = metadata_hf['split'].fillna('').astype(str)
metadata_hf['duration'] = metadata_hf['duration'].astype(float)

features = Features(
    {
        'audio': Audio(sampling_rate=16000),
        'resource': Value('string'),
        'path': Value('string'),
        'sentence': Value('string'),
        'duration': Value('float64'),
        'source_id': Value('string'),
        'duration_bin': Value('string'),
        'split': Value('string')
    }
)

dataset_dict = DatasetDict()

for split in ['train', 'dev', 'test']:
    split_df = metadata_hf[metadata_hf['split'] == split].copy()

    if len(split_df) == 0:
        continue

    records = split_df.to_dict('records')

    ds = Dataset.from_list(
        records,
        features=features
    )

    dataset_dict[split] = ds

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 2714
    })
    dev: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 140
    })
    test: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 429
    })
})

In [ ]:
example = dataset_dict['train'][-1]

print(example.keys())
print(example['resource'])
print(example['path'])
print(example['sentence'])
print(example['audio'])

dict_keys(['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'])
bible
Luke_024_segment-8.wav
экык о'равэтԓьаргэн йыԓьёԓӄыԓ мынгэты э'ӄэԓтэтыԓьыргин ы'ԓӄаптымъёԓӄыԓ ынкъам ӈыроӄав ы'ԓёк ытԓён эюԓьыԓӄыԓ


In [ ]:
print(example['audio']['sampling_rate'])
print(example['audio']['array'].shape)

16000
(152160,)


In [ ]:
repo_id = 'tadgeis/chukchi-asr-data-private'
create_repo(repo_id=repo_id, repo_type='dataset', private=True, exist_ok=True)

RepoUrl('https://huggingface.co/datasets/tadgeis/chukchi-asr-data-private', endpoint='https://huggingface.co', repo_type='dataset', repo_id='tadgeis/chukchi-asr-data-private')

In [ ]:
dataset_dict.push_to_hub(repo_id, private=True, max_shard_size='500MB')

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Map:   0%|          | 0/1357 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpi8yuvl5h.parquet    :   3%|3         | 4.70MB /  145MB            

CommitInfo(commit_url='https://huggingface.co/datasets/tadgeis/chukchi-asr-data-private/commit/ed839e19203dbbbbf168a5fd09e8cf052fd30525', commit_message='Upload dataset', commit_description='', oid='ed839e19203dbbbbf168a5fd09e8cf052fd30525', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/tadgeis/chukchi-asr-data-private', endpoint='https://huggingface.co', repo_type='dataset', repo_id='tadgeis/chukchi-asr-data-private'), pr_revision=None, pr_num=None)

In [ ]:
readme_text = """---
pretty_name: Private Chukchi ASR Dataset
task_categories:
- automatic-speech-recognition
language:
- ckt
private: true
---

# Private Chukchi ASR Dataset

This is a private dataset for Chukchi ASR experiments.

It contains segmented audio and transcripts from three resources:

- bible
- radio "Purga"
- chuklang expedition data

The dataset is not intended for public redistribution. Access should remain restricted.
"""

Path('README.md').write_text(readme_text, encoding='utf-8')

api = HfApi()

api.upload_file(path_or_fileobj='README.md', path_in_repo='README.md',
    repo_id=repo_id, repo_type='dataset')

CommitInfo(commit_url='https://huggingface.co/datasets/tadgeis/chukchi-asr-data-private/commit/78d016a342cf7b1f3c74705e683cc84753e9d4dc', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='78d016a342cf7b1f3c74705e683cc84753e9d4dc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/tadgeis/chukchi-asr-data-private', endpoint='https://huggingface.co', repo_type='dataset', repo_id='tadgeis/chukchi-asr-data-private'), pr_revision=None, pr_num=None)

# MMS existing Chukchi adapter
using code parts from https://huggingface.co/docs/transformers/en/model_doc/mms

In [ ]:
!pip install -U transformers accelerate datasets[audio] jiwer pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 30.9 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.8.0.dev0
    Uninstalling transformers-5.8.0.dev0:
      Successfully uninstalled transformers-5.8.0.dev0


In [ ]:
from google.colab import userdata
from huggingface_hub import login

from datasets import Audio, load_dataset

import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC

from tqdm.auto import tqdm
import jiwer
import pandas as pd

import re

In [ ]:
hf_token = userdata.get('HF_TOKEN')

if hf_token is None:
    raise ValueError('HF_TOKEN not found in Colab Secrets')

login(token=hf_token)

In [ ]:
repo_id = 'tadgeis/chukchi-asr-data-private'

dataset_dict = load_dataset(repo_id, token=hf_token, download_mode='force_redownload')
dataset_dict = dataset_dict.cast_column('audio', Audio(sampling_rate=16000))

dataset_dict

data/train-00000-of-00002.parquet:   0%|          | 0.00/411M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/410M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/145M [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/60.0M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating dev split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 2714
    })
    test: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 429
    })
    dev: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 140
    })
})

In [ ]:
example = dataset_dict['train'][-1]

print(example.keys())
print(example['resource'])
print(example['path'])
print(example['sentence'])
print(example['audio']['sampling_rate'])
print(example['audio']['array'].shape)

dict_keys(['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'])
bible
Luke_024_segment-8.wav
экык о'равэтԓьаргэн йыԓьёԓӄыԓ мынгэты э'ӄэԓтэтыԓьыргин ы'ԓӄаптымъёԓӄыԓ ынкъам ӈыроӄав ы'ԓёк ытԓён эюԓьыԓӄыԓ
16000
(152160,)


In [ ]:
chuklang_test = dataset_dict['test'].filter(lambda example: example['resource'] == 'chuklang')
radio_test = dataset_dict['test'].filter(lambda example: example['resource'] == 'radio')
bible_test = dataset_dict['test'].filter(lambda example: example['resource'] == 'bible')

chuklang_test, radio_test, bible_test

Filter:   0%|          | 0/429 [00:00<?, ? examples/s]

Filter:   0%|          | 0/429 [00:00<?, ? examples/s]

Filter:   0%|          | 0/429 [00:00<?, ? examples/s]

(Dataset({
     features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
     num_rows: 200
 }),
 Dataset({
     features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
     num_rows: 83
 }),
 Dataset({
     features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
     num_rows: 146
 }))

In [ ]:
model_id = 'facebook/mms-1b-all'
target_lang = 'ckt'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

processor = AutoProcessor.from_pretrained(model_id, target_lang=target_lang)
model = Wav2Vec2ForCTC.from_pretrained(model_id, target_lang=target_lang,
    ignore_mismatched_sizes=True, device_map='auto')

model = model.to(device)
model.eval()

Device: cpu


NameError: name 'AutoProcessor' is not defined

In [ ]:
vocab = processor.tokenizer.get_vocab()

print('Vocab size:', len(vocab))

for token, token_id in sorted(vocab.items(), key=lambda item: item[1]):
    print(token_id, repr(token))

Vocab size: 42
0 '<pad>'
1 '<s>'
2 '</s>'
3 '<unk>'
4 'ы'
5 '|'
6 'н'
7 'э'
8 'т'
9 'а'
10 'и'
11 'к'
12 'р'
13 'ԓ'
14 'м'
15 'г'
16 'в'
17 'ъ'
18 'ӈ'
19 'ӄ'
20 'о'
21 'у'
22 'ь'
23 'ч'
24 'е'
25 'й'
26 'п'
27 'ё'
28 'с'
29 'я'
30 '-'
31 '–'
32 'ю'
33 'л'
34 "'"
35 'д'
36 'ф'
37 'з'
38 'х'
39 'б'
40 'ж'
41 'ц'


In [ ]:
sample = chuklang_test[0]['audio']['array']

inputs = processor(sample, sampling_rate=16_000, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model(**inputs).logits

ids = torch.argmax(outputs, dim=-1)[0]
transcription = processor.decode(ids)

print('reference:')
print(chuklang_test[0]['sentence'])

print()
print('prediction:')
print(transcription)

reference:
ӄоле итгъэт ӄынвэтэ ӈиръэ ӈэвысӄэтти элерэты натанат

prediction:
ыӄоԓе итъатӄынвытэ ниръэӈ эвысӄаттэ эԓерэтынатанат


In [ ]:
def transcribe_dataset_mms(dataset, batch_size=4):
    predictions = []
    references = []
    paths = []
    resources = []

    for start in tqdm(range(0, len(dataset), batch_size)):
        batch = dataset[start:start + batch_size]

        audio_arrays = [audio_item['array'] for audio_item in batch['audio']]

        inputs = processor(audio_arrays, sampling_rate=16_000,
            return_tensors='pt', padding=True).to(model.device)

        with torch.no_grad():
            logits = model(**inputs).logits

        predicted_ids = torch.argmax(logits, dim=-1)
        batch_predictions = processor.batch_decode(predicted_ids)

        predictions.extend(batch_predictions)
        references.extend(batch['sentence'])
        paths.extend(batch['path'])
        resources.extend(batch['resource'])

    return {
        'resource': resources,
        'path': paths,
        'reference': references,
        'prediction': predictions
    }

In [ ]:
def normalize_spaces(text):
    if pd.isna(text):
        return ''

    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text


def evaluate_subset(dataset, subset_name, batch_size=4):
    results = transcribe_dataset_mms(dataset, batch_size=batch_size)

    df = pd.DataFrame(results)

    df['reference'] = df['reference'].apply(normalize_spaces)
    df['prediction'] = df['prediction'].apply(normalize_spaces)

    wer = jiwer.wer(
        df['reference'].tolist(),
        df['prediction'].tolist()
    )

    cer = jiwer.cer(
        df['reference'].tolist(),
        df['prediction'].tolist()
    )

    print(f'{subset_name} WER: {wer:.4f}')
    print(f'{subset_name} CER: {cer:.4f}')

    df.to_csv(f'mms_existing_ckt_{subset_name}_predictions.csv',
        index=False, encoding='utf-8-sig')

    return df, wer, cer

In [ ]:
chuklang_df, chuklang_wer, chuklang_cer = evaluate_subset(chuklang_test, 'chuklang_test', batch_size=4)

radio_df, radio_wer, radio_cer = evaluate_subset(radio_test, 'radio_test', batch_size=4)

bible_df, bible_wer, bible_cer = evaluate_subset(bible_test, 'bible_test', batch_size=4)

  0%|          | 0/50 [00:00<?, ?it/s]

chuklang_test WER: 0.9236
chuklang_test CER: 0.2999


  0%|          | 0/21 [00:00<?, ?it/s]

radio_test WER: 0.9159
radio_test CER: 0.3313


  0%|          | 0/37 [00:00<?, ?it/s]

bible_test WER: 0.5245
bible_test CER: 0.0754


In [ ]:
results_summary = pd.DataFrame(
    [
        {
            'model': 'MMS-1b-all existing ckt adapter',
            'subset': 'chuklang_test',
            'WER': chuklang_wer,
            'CER': chuklang_cer,
            'n_files': len(chuklang_df)
        },
        {
            'model': 'MMS-1b-all existing ckt adapter',
            'subset': 'radio_test',
            'WER': radio_wer,
            'CER': radio_cer,
            'n_files': len(radio_df)
        },
        {
            'model': 'MMS-1b-all existing ckt adapter',
            'subset': 'bible_test',
            'WER': bible_wer,
            'CER': bible_cer,
            'n_files': len(bible_df)
        }
    ]
)

results_summary

,model,subset,WER,CER,n_files
0,MMS-1b-all existing ckt adapter,chuklang_test,0.923619,0.299876,200
1,MMS-1b-all existing ckt adapter,radio_test,0.915888,0.331299,83
2,MMS-1b-all existing ckt adapter,bible_test,0.524542,0.075359,146


In [ ]:
results_summary.to_csv('mms_existing_ckt_results_summary.csv',
                      index=False, encoding='utf-8-sig')

# Models comparison

In [ ]:
import pandas as pd

In [ ]:
df_mms_existing_ckt = pd.read_csv('/content/mms_existing_ckt_results_summary.csv')
df_mms_adapter_chuklang_only = pd.read_csv('/content/mms_adapter_chuklang_only_results_summary.csv')
df_mms_adapter_pooled = pd.read_csv('/content/mms_adapter_pooled_results_summary.csv')
df_mms_adapter_staged_bible_radio_to_chuklang_stage2 = pd.read_csv('/content/mms_adapter_staged_bible_radio_to_chuklang_stage2_results_summary.csv')
df_mms_adapter_staged_bible_radio_to_chuklang_source_eval_stage2 = pd.read_csv('/content/mms_adapter_staged_bible_radio_to_chuklang_source_eval_stage2_results_summary.csv')

df_xlsr_1b_chuklang_only = pd.read_csv('/content/xlsr_1b_chuklang_only_results_summary.csv')
df_xlsr_1b_pooled_to_chuklang = pd.read_csv('/content/xlsr_1b_pooled_to_chuklang_results_summary.csv')
df_xlsr_1b_pooled_to_chuklang_stage2_chuklang_only = pd.read_csv('/content/xlsr_1b_pooled_to_chuklang_stage2_chuklang_only_results_summary.csv')
df_xlsr_1b_staged_bible_radio_to_chuklang_source_eval = pd.read_csv('/content/xlsr_1b_staged_bible_radio_to_chuklang_source_eval_results_summary.csv')
df_xlsr_300m_staged_bible_radio_to_chuklang_source_eval = pd.read_csv('/content/xlsr_300m_staged_bible_radio_to_chuklang_source_eval_results_summary.csv')

In [ ]:
results = pd.concat([df_mms_existing_ckt, df_mms_adapter_chuklang_only, df_mms_adapter_pooled, df_mms_adapter_staged_bible_radio_to_chuklang_stage2,
          df_mms_adapter_staged_bible_radio_to_chuklang_source_eval_stage2, df_xlsr_1b_chuklang_only, df_xlsr_1b_pooled_to_chuklang,
          df_xlsr_1b_pooled_to_chuklang_stage2_chuklang_only, df_xlsr_1b_staged_bible_radio_to_chuklang_source_eval,
          df_xlsr_300m_staged_bible_radio_to_chuklang_source_eval])
results = results.reset_index(drop = True)

indices_to_drop = []

indices_to_drop.extend(results[(results.training == 'staged_bible_radio_to_chuklang_source_eval_stage2') & (results.subset != 'chuklang_test')].index)
indices_to_drop.extend(results[(results.training == 'xlsr_1b_staged_bible_radio_to_chuklang_source_eval')  & (results.subset != 'chuklang_test')].index)
indices_to_drop.extend(results[(results.training == 'xlsr_300m_staged_bible_radio_to_chuklang_source_eval')  & (results.subset != 'chuklang_test')].index)

results.drop(indices_to_drop, inplace=True)

In [ ]:
results.to_csv('results.csv', encoding='utf-8-sig')

In [ ]:
results = pd.read_csv('/content/results.csv')

In [ ]:
results_chuklang_test = results[results.subset == 'chuklang_test']
results_chuklang_test.sort_values(by='CER')

,Unnamed: 0,model,subset,WER,CER,n_files,training,predictions_file,best_dev_checkpoint,best_dev_step,...,stage1_best_dev_checkpoint,stage1_best_dev_step,stage1_best_dev_loss,stage1_best_dev_WER,stage1_best_dev_CER,stage2_best_dev_checkpoint,stage2_best_dev_step,stage2_best_dev_loss,stage2_best_dev_WER,stage2_best_dev_CER
12,12,MMS-1b-all adapter fine-tuning,chuklang_test,0.739130,0.182521,200,staged_bible_radio_to_chuklang_source_eval_stage2,mms_adapter_staged_bible_radio_to_chuklang_sou...,NaN,NaN,...,mms-1b-ckt-staged-bible-radio-to-chuklang-sour...,950.0,0.266897,0.347951,0.061646,mms-1b-ckt-staged-bible-radio-to-chuklang-sour...,550.0,0.660495,0.671429,0.158405
9,9,MMS-1b-all adapter fine-tuning,chuklang_test,0.733255,0.184316,200,staged_bible_radio_to_chuklang_stage2,mms_adapter_staged_bible_radio_to_chuklang_sta...,NaN,NaN,...,mms-1b-ckt-staged-bible-radio-to-chuklang-stag...,150.0,1.349748,0.938095,0.329248,mms-1b-ckt-staged-bible-radio-to-chuklang-stag...,750.0,0.672790,0.712698,0.164624
3,3,MMS-1b-all adapter fine-tuning,chuklang_test,0.768508,0.195361,200,chuklang_only,mms_adapter_chuklang_only_chuklang_test_predic...,mms-1b-ckt-chuklang-only/checkpoint-650,650.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,19,XLS-R 1B pooled CTC fine-tuning + chuklang-onl...,chuklang_test,0.777908,0.205440,200,xlsr_1b_pooled_to_chuklang_stage2_chuklang_only,xlsr_1b_pooled_to_chuklang_stage2_chuklang_onl...,xlsr-1b-pooled-to-chuklang-stage2-chuklang/che...,200.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,18,XLS-R 1B pooled CTC fine-tuning,chuklang_test,0.768508,0.206820,200,xlsr_1b_pooled_to_chuklang,xlsr_1b_pooled_to_chuklang_chuklang_test_predi...,xlsr-1b-pooled-to-chuklang/checkpoint-2700,2700.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,23,XLS-R 300M CTC fine-tuning,chuklang_test,0.866040,0.253624,200,xlsr_300m_staged_bible_radio_to_chuklang_sourc...,xlsr_300m_staged_bible_radio_to_chuklang_sourc...,NaN,NaN,...,xls-r-300m-ckt-staged-bible-radio-to-chuklang-...,2600.0,0.358144,0.434426,0.084442,xls-r-300m-ckt-staged-bible-radio-to-chuklang-...,100.0,1.173240,0.863492,0.240534
6,6,MMS-1b-all adapter fine-tuning,chuklang_test,0.901293,0.255833,200,pooled,mms_adapter_pooled_chuklang_test_predictions.csv,mms-1b-ckt-pooled/checkpoint-150,150.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,20,XLS-R 1B CTC fine-tuning,chuklang_test,0.886016,0.275024,200,xlsr_1b_staged_bible_radio_to_chuklang_source_...,xlsr_1b_staged_bible_radio_to_chuklang_source_...,NaN,NaN,...,xls-r-1b-ckt-staged-bible-radio-to-chuklang-so...,1800.0,0.261868,0.269672,0.050585,xls-r-1b-ckt-staged-bible-radio-to-chuklang-so...,550.0,1.070491,0.857143,0.255716
0,0,MMS-1b-all existing ckt adapter,chuklang_test,0.923619,0.299876,200,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,15,XLS-R 1B CTC fine-tuning,chuklang_test,0.970623,0.333287,200,xlsr_1b_chuklang_only,xlsr_1b_chuklang_only_chuklang_test_prediction...,xlsr-1b-ckt-chuklang-only/checkpoint-1150,1150.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Create HF repository for best model

In [ ]:
!pip -q install huggingface_hub

In [ ]:
from huggingface_hub import notebook_login

from pathlib import Path

from huggingface_hub import HfApi, snapshot_download, model_info

In [ ]:
notebook_login()
api = HfApi()

In [ ]:
source_repo_id = 'tadgeis/mms-1b-ckt-staged-bible-radio-to-chuklang-source-eval-stage2'
new_repo_id = 'tadgeis/mms-1b-all-ckt-best-cer-model'

local_dir = '/content/mms-1b-all-ckt-best-cer-model'

In [ ]:
api.create_repo(
    repo_id=new_repo_id,
    repo_type='model',
    private=False,
    exist_ok=True
)

RepoUrl('https://huggingface.co/tadgeis/mms-1b-all-ckt-best-cer-model', endpoint='https://huggingface.co', repo_type='model', repo_id='tadgeis/mms-1b-all-ckt-best-cer-model')

In [ ]:
needed_files = [
    'model.safetensors',
    'adapter.ckt.safetensors',
    'config.json',
    'preprocessor_config.json',
    'tokenizer_config.json',
    'vocab.json',
    'special_tokens_map.json',
    'added_tokens.json',
    'README.md',
    '.gitattributes'
]

snapshot_download(
    repo_id=source_repo_id,
    repo_type='model',
    local_dir=local_dir,
    allow_patterns=needed_files
)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

'/content/mms-1b-all-ckt-best-cer-model'

In [ ]:
api.upload_folder(
    folder_path=local_dir,
    repo_id=new_repo_id,
    repo_type='model',
    commit_message='Upload cleaned best CER MMS CKT model for inference'
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...l/adapter.ckt.safetensors:  62%|######2   | 5.51MB / 8.85MB            

  ...r-model/model.safetensors:   1%|          | 31.9MB / 3.86GB            

CommitInfo(commit_url='https://huggingface.co/tadgeis/mms-1b-all-ckt-best-cer-model/commit/8522bc190eb767a9fd993d90a001e85f922ca174', commit_message='Upload cleaned best CER MMS CKT model for inference', commit_description='', oid='8522bc190eb767a9fd993d90a001e85f922ca174', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tadgeis/mms-1b-all-ckt-best-cer-model', endpoint='https://huggingface.co', repo_type='model', repo_id='tadgeis/mms-1b-all-ckt-best-cer-model'), pr_revision=None, pr_num=None)

In [ ]:
def human_size(num_bytes):
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if num_bytes < 1024:
            return f'{num_bytes:.2f} {unit}'
        num_bytes /= 1024

    return f'{num_bytes:.2f} PB'


def show_hf_repo_files(repo_id):
    info = model_info(repo_id, files_metadata=True)

    rows = []

    for file_info in info.siblings:
        size = file_info.size or 0

        rows.append({
            'filename': file_info.rfilename,
            'size': size,
            'size_readable': human_size(size)
        })

    rows = sorted(rows, key=lambda row: row['size'], reverse=True)

    total_size = sum(row['size'] for row in rows)

    print('Total:', human_size(total_size))

    for row in rows:
        print(row['size_readable'], row['filename'])

    return rows


repo_files = show_hf_repo_files(new_repo_id)

Total: 3.60 GB
3.59 GB model.safetensors
8.44 MB adapter.ckt.safetensors
2.41 KB README.md
1.95 KB config.json
1.48 KB .gitattributes
1.10 KB tokenizer_config.json
572.00 B vocab.json
256.00 B preprocessor_config.json
96.00 B special_tokens_map.json
30.00 B added_tokens.json


# Testing statistical criteria

# Chuklang cyrillic and IPA matching

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd
import unicodedata
import re

import zipfile
import shutil
import os

In [ ]:
eaf_dir = Path('chuklang_eafs')

metadata_resource_col = 'resource'
metadata_path_col = 'path'
metadata_text_col = 'sentence'

target_tier_id = 'transcription'

In [ ]:
metadata = pd.read_csv('metadata.tsv', sep='\t', encoding='utf-8-sig')
metadata.head()

,resource,path,sentence,duration
0,radio,11_01-18.00_segment-10.wav,ымыԓьо кэлит тыӈивылӄылти ӄликкин январьтагнэты,3.300000
1,radio,11_01-18.00_segment-11.wav,миӈкыри кэлийвылӄылти рыкэтъоӈаннэн инэнлеԓьын...,8.967982
2,radio,11_01-18.00_segment-12.wav,ынан тывнэн нымытваԓьыт микынти гэнъэллинэт эн...,17.503129
3,radio,11_01-18.00_segment-13.wav,чама микынти эннукэ гитлинэт нэмыӄэй рыԓьататъ...,6.080998
4,radio,11_01-18.00_segment-14.wav,отчётак кэлийвылӄыл чама миӈкы ръаынныӈыттынвы...,6.933016


In [ ]:
zip_path = Path(f'{eaf_dir}.zip')
extract_dir = Path(eaf_dir)

if extract_dir.exists():
    shutil.rmtree(extract_dir)

extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f'Extracted to: {extract_dir}')

Extracted to: chuklang_eafs


In [ ]:
def remove_punctuation_keep_apostrophes(text):
    chars = []

    for char in text:
        category = unicodedata.category(char)

        if category.startswith('P') and char not in "'":
            chars.append(' ')
        else:
            chars.append(char)

    return ''.join(chars)


def normalize_text_basic(text):
    if pd.isna(text):
        return ''

    text = str(text)
    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    text = text.replace('\u00a0', ' ')
    text = remove_punctuation_keep_apostrophes(text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def normalize_recording_key(value):
    if pd.isna(value):
        return ''

    value = str(value)
    value = unicodedata.normalize('NFC', value)
    value = Path(value).stem
    value = value.lower()
    value = value.replace('\u00a0', ' ')

    value = re.sub(r'[_]+[0-9]+$', '', value)

    return value


def get_recording_id_from_metadata_path(path):
    stem = Path(str(path)).stem
    stem = re.sub(r'[_\-\s]+[0-9]+$', '', stem)

    return stem


def get_segment_number_from_metadata_path(path):
    stem = Path(str(path)).stem
    match = re.search(r'[_\-\s]+([0-9]+)$', stem)

    if match is None:
        return None

    return int(match.group(1))


def find_eaf_paths(eaf_dir):
    eaf_dir = Path(eaf_dir)

    return sorted(
        path for path in eaf_dir.rglob('*')
        if path.is_file() and path.suffix.lower() == '.eaf'
    )


def local_name(tag):
    return tag.split('}', 1)[-1] if '}' in tag else tag


def iter_by_tag(root, tag_name):
    for elem in root.iter():
        if local_name(elem.tag) == tag_name:
            yield elem


def find_first_by_tag(elem, tag_name):
    for child in elem.iter():
        if local_name(child.tag) == tag_name:
            return child

    return None


def annotation_value_text(annotation_elem):
    value_elem = find_first_by_tag(annotation_elem, 'ANNOTATION_VALUE')

    if value_elem is None:
        return ''

    return ''.join(value_elem.itertext()).strip()


def parse_eaf(eaf_path):
    eaf_path = Path(eaf_path)

    tree = ET.parse(eaf_path)
    root = tree.getroot()

    time_slots = {}

    for elem in iter_by_tag(root, 'TIME_SLOT'):
        time_slot_id = elem.attrib.get('TIME_SLOT_ID')
        time_value = elem.attrib.get('TIME_VALUE')

        if time_slot_id is None:
            continue

        if time_value is None:
            time_slots[time_slot_id] = None
        else:
            time_slots[time_slot_id] = int(time_value)

    rows = []

    for tier in iter_by_tag(root, 'TIER'):
        tier_id = tier.attrib.get('TIER_ID')
        linguistic_type = tier.attrib.get('LINGUISTIC_TYPE_REF')
        participant = tier.attrib.get('PARTICIPANT')
        annotator = tier.attrib.get('ANNOTATOR')

        for annotation in tier:
            if local_name(annotation.tag) != 'ANNOTATION':
                continue

            inner = None

            for child in annotation:
                child_name = local_name(child.tag)

                if child_name in {'ALIGNABLE_ANNOTATION', 'REF_ANNOTATION'}:
                    inner = child
                    break

            if inner is None:
                continue

            inner_type = local_name(inner.tag)
            annotation_id = inner.attrib.get('ANNOTATION_ID')
            parent_id = inner.attrib.get('ANNOTATION_REF')
            text = annotation_value_text(inner)

            if inner_type == 'ALIGNABLE_ANNOTATION':
                start_ref = inner.attrib.get('TIME_SLOT_REF1')
                end_ref = inner.attrib.get('TIME_SLOT_REF2')

                start_ms = time_slots.get(start_ref)
                end_ms = time_slots.get(end_ref)
            else:
                start_ms = None
                end_ms = None

            rows.append({
                'eaf_path': str(eaf_path),
                'eaf_file': eaf_path.name,
                'recording_id': eaf_path.stem,
                'tier_id': tier_id,
                'linguistic_type': linguistic_type,
                'participant': participant,
                'annotator': annotator,
                'annotation_id': annotation_id,
                'parent_id': parent_id,
                'annotation_type': inner_type,
                'start_ms': start_ms,
                'end_ms': end_ms,
                'text': text,
            })

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    annotations_by_id = {
        row['annotation_id']: row
        for _, row in df.iterrows()
        if pd.notna(row['annotation_id'])
    }

    def resolve_time(annotation_id, field, seen=None):
        if seen is None:
            seen = set()

        if annotation_id in seen:
            return None

        seen.add(annotation_id)

        row = annotations_by_id.get(annotation_id)

        if row is None:
            return None

        value = row[field]

        if pd.notna(value):
            return int(value)

        parent_id = row['parent_id']

        if pd.isna(parent_id):
            return None

        return resolve_time(parent_id, field, seen)

    for idx, row in df.iterrows():
        if pd.isna(row['start_ms']) and pd.notna(row['annotation_id']):
            df.at[idx, 'start_ms'] = resolve_time(row['annotation_id'], 'start_ms')

        if pd.isna(row['end_ms']) and pd.notna(row['annotation_id']):
            df.at[idx, 'end_ms'] = resolve_time(row['annotation_id'], 'end_ms')

    df['start_ms'] = pd.to_numeric(df['start_ms'], errors='coerce').astype('Int64')
    df['end_ms'] = pd.to_numeric(df['end_ms'], errors='coerce').astype('Int64')
    df['duration_ms'] = df['end_ms'] - df['start_ms']

    return df


def normalize_tier_id(tier_id):
    if pd.isna(tier_id):
        return ''

    tier_id = str(tier_id)
    tier_id = unicodedata.normalize('NFC', tier_id)
    tier_id = tier_id.lower()
    tier_id = tier_id.replace('\u00a0', ' ')
    tier_id = re.sub(r'\s+', ' ', tier_id).strip()

    return tier_id


def is_transcription_tier(tier_id):
    tier_id_norm = normalize_tier_id(tier_id)

    return bool(re.fullmatch(r'transcription[\s_\-]*\d*', tier_id_norm))


def get_transcription_tier_order(tier_id):
    tier_id_norm = normalize_tier_id(tier_id)
    match = re.search(r'(\d+)$', tier_id_norm)

    if match is None:
        return 0

    return int(match.group(1))


def collect_eaf_transcriptions(eaf_dir):
    eaf_paths = find_eaf_paths(eaf_dir)
    rows = []

    for eaf_path in eaf_paths:
        df = parse_eaf(eaf_path)

        if df.empty:
            continue

        df['tier_id_norm'] = df['tier_id'].apply(normalize_tier_id)
        df['is_transcription_tier'] = df['tier_id'].apply(is_transcription_tier)
        df['transcription_tier_order'] = df['tier_id'].apply(get_transcription_tier_order)

        selected = df[df['is_transcription_tier']].copy()
        selected = selected[selected['text'].astype(str).str.strip().ne('')].copy()

        if selected.empty:
            continue

        selected['eaf_recording_id'] = eaf_path.stem
        selected['recording_key'] = selected['eaf_recording_id'].apply(normalize_recording_key)

        rows.append(selected)

    if not rows:
        return pd.DataFrame()

    result = pd.concat(rows, ignore_index=True)

    result = result.sort_values(
        [
            'recording_key',
            'start_ms',
            'end_ms',
            'transcription_tier_order',
            'tier_id_norm'
        ],
        na_position='last'
    ).reset_index(drop=True)

    result['ipa_original'] = result['text']
    result['ipa_norm'] = result['ipa_original'].apply(normalize_text_basic)

    return result


def aggregate_eaf_by_recording(eaf_df):
    df = eaf_df.copy()

    if df.empty:
        return pd.DataFrame()

    df = df[df['ipa_original'].astype(str).str.strip().ne('')]

    df = df.sort_values(
        [
            'recording_key',
            'start_ms',
            'end_ms',
            'transcription_tier_order',
            'tier_id_norm'
        ],
        na_position='last'
    )

    result = (
        df
        .groupby('recording_key')
        .agg(
            eaf_recording_id=('eaf_recording_id', 'first'),
            ipa_full=('ipa_original', lambda x: ' '.join(x.astype(str))),
            n_eaf_annotations=('ipa_original', 'size'),
            eaf_tiers=('tier_id', lambda x: sorted(set(x.dropna().astype(str)))),
            eaf_start_ms=('start_ms', 'min'),
            eaf_end_ms=('end_ms', 'max')
        )
        .reset_index()
    )

    result['ipa_full_norm'] = result['ipa_full'].apply(normalize_text_basic)

    return result


def prepare_chuklang_metadata(metadata):
    df = metadata.copy()

    df = df[df[metadata_resource_col] == 'chuklang'].copy()

    df['metadata_recording_id'] = df[metadata_path_col].apply(get_recording_id_from_metadata_path)
    df['recording_key'] = df[metadata_path_col].apply(normalize_recording_key)
    df['segment_number'] = df[metadata_path_col].apply(get_segment_number_from_metadata_path)

    df['cyrillic_original'] = df[metadata_text_col].fillna('').astype(str)
    df['cyrillic_norm'] = df['cyrillic_original'].apply(normalize_text_basic)

    return df


def aggregate_metadata_by_recording(chuklang_metadata):
    df = chuklang_metadata.copy()

    df = df[df['cyrillic_original'].astype(str).str.strip().ne('')]
    df['_segment_sort'] = df['segment_number'].fillna(10**12)

    df = df.sort_values(['recording_key', '_segment_sort', metadata_path_col])

    result = (
        df
        .groupby('recording_key')
        .agg(
            metadata_recording_id=('metadata_recording_id', 'first'),
            cyrillic_full=('cyrillic_original', lambda x: ' '.join(x.astype(str))),
            n_metadata_segments=('cyrillic_original', 'size'),
            segment_numbers=('segment_number', lambda x: list(x.dropna().astype(int)))
        )
        .reset_index()
    )

    result['cyrillic_full_norm'] = result['cyrillic_full'].apply(normalize_text_basic)

    return result

In [ ]:
eaf_transcriptions = collect_eaf_transcriptions(eaf_dir)

eaf_full = aggregate_eaf_by_recording(eaf_transcriptions)

print(f'EAF recordings: {len(eaf_full)}')
eaf_full.head()

EAF recordings: 65


,recording_key,eaf_recording_id,ipa_full,n_eaf_annotations,eaf_tiers,eaf_start_ms,eaf_end_ms,ipa_full_norm
0,a chatterbox and a wanton girl,A chatterbox and a wanton girl,təŋeɬqotna nenatwəqen. qoɬe itɣʔet qənwete ŋir...,9,[transcription],904,47260,təŋeɬqotna nenatwəqen qoɬe itɣʔet qənwete ŋirʔ...
1,abramovich,Abramovich,kətur ənŋəteqe medwedew prezidento enma kančal...,6,[transcription],375,42370,kətur ənŋəteqe medwedew prezidento enma kančal...
2,an evil spirit and a dicky bird,An evil spirit and a dicky bird,enmen ɣatwaɬen kaɬʔajŋən ənkʔam pseqaɬɣən. ɬʔu...,14,[transcription],660,77500,enmen ɣatwaɬen kaɬʔajŋən ənkʔam pseqaɬɣən ɬʔuw...
3,being a child,Being a child,qeɬoqʔəm emnuŋkə ɣəm ɣʔuretiɣəme. ətɬʔataŋ emn...,33,[transcription],0,273637,qeɬoqʔəm emnuŋkə ɣəm ɣʔuretiɣəme ətɬʔataŋ emnu...
4,birth,Birth,ɣəm emnuŋkə tʔuretɣʔek. ɣəmninet ətɬʔat ɣəmnin...,9,[transcription],400,81090,ɣəm emnuŋkə tʔuretɣʔek ɣəmninet ətɬʔat ɣəmnin ...


In [ ]:
chuklang_metadata = prepare_chuklang_metadata(metadata)

metadata_full = aggregate_metadata_by_recording(chuklang_metadata)

print(f'Metadata recordings: {len(metadata_full)}')
metadata_full.head()

Metadata recordings: 65


,recording_key,metadata_recording_id,cyrillic_full,n_metadata_segments,segment_numbers,cyrillic_full_norm
0,a chatterbox and a wanton girl,A chatterbox and a wanton girl,тыӈэлӄотна нэнатвыӄэн ӄоле итгъэт ӄынвэтэ ӈиръ...,9,"[1, 2, 3, 4, 5, 6, 7, 8, 9]",тыӈэлӄотна нэнатвыӄэн ӄоле итгъэт ӄынвэтэ ӈиръ...
1,abramovich,Abramovich,кытур ынӈытэӄэ мэдвэдэв прэзидэнто энма канчал...,6,"[1, 2, 3, 4, 5, 6]",кытур ынӈытэӄэ мэдвэдэв прэзидэнто энма канчал...
2,an evil spirit and a dicky bird,An evil spirit and a dicky bird,энмэн гатвален каԓьайӈын ынкъам пчеӄалгын ԓьув...,14,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]",энмэн гатвален каԓьайӈын ынкъам пчеӄалгын ԓьув...
3,being a child,Being a child,ӄэлёӄы'м эмнуӈкы гым гъурэтигымэ ытԓьатаӈ эмну...,33,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",ӄэлёӄы'м эмнуӈкы гым гъурэтигымэ ытԓьатаӈ эмну...
4,birth,Birth,гым эмнуӈкы тъурэтгъэк гымнинэт ытԓьат гымнин ...,9,"[1, 2, 3, 4, 5, 6, 7, 8, 9]",гым эмнуӈкы тъурэтгъэк гымнинэт ытԓьат гымнин ...


In [ ]:
full_pairs = metadata_full.merge(eaf_full, on='recording_key', how='outer', indicator=True)

usable_pairs = full_pairs[full_pairs['_merge'] == 'both'].copy()

usable_pairs = usable_pairs[
    [
        'recording_key',
        'metadata_recording_id',
        'eaf_recording_id',
        'cyrillic_full',
        'cyrillic_full_norm',
        'ipa_full',
        'ipa_full_norm',
        'n_metadata_segments',
        'n_eaf_annotations',
        'segment_numbers',
        'eaf_tiers',
        'eaf_start_ms',
        'eaf_end_ms'
    ]
].copy()

usable_pairs.head()

,recording_key,metadata_recording_id,eaf_recording_id,cyrillic_full,cyrillic_full_norm,ipa_full,ipa_full_norm,n_metadata_segments,n_eaf_annotations,segment_numbers,eaf_tiers,eaf_start_ms,eaf_end_ms
0,a chatterbox and a wanton girl,A chatterbox and a wanton girl,A chatterbox and a wanton girl,тыӈэлӄотна нэнатвыӄэн ӄоле итгъэт ӄынвэтэ ӈиръ...,тыӈэлӄотна нэнатвыӄэн ӄоле итгъэт ӄынвэтэ ӈиръ...,təŋeɬqotna nenatwəqen. qoɬe itɣʔet qənwete ŋir...,təŋeɬqotna nenatwəqen qoɬe itɣʔet qənwete ŋirʔ...,9,9,"[1, 2, 3, 4, 5, 6, 7, 8, 9]",[transcription],904,47260
1,abramovich,Abramovich,Abramovich,кытур ынӈытэӄэ мэдвэдэв прэзидэнто энма канчал...,кытур ынӈытэӄэ мэдвэдэв прэзидэнто энма канчал...,kətur ənŋəteqe medwedew prezidento enma kančal...,kətur ənŋəteqe medwedew prezidento enma kančal...,6,6,"[1, 2, 3, 4, 5, 6]",[transcription],375,42370
2,an evil spirit and a dicky bird,An evil spirit and a dicky bird,An evil spirit and a dicky bird,энмэн гатвален каԓьайӈын ынкъам пчеӄалгын ԓьув...,энмэн гатвален каԓьайӈын ынкъам пчеӄалгын ԓьув...,enmen ɣatwaɬen kaɬʔajŋən ənkʔam pseqaɬɣən. ɬʔu...,enmen ɣatwaɬen kaɬʔajŋən ənkʔam pseqaɬɣən ɬʔuw...,14,14,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]",[transcription],660,77500
3,being a child,Being a child,Being a child,ӄэлёӄы'м эмнуӈкы гым гъурэтигымэ ытԓьатаӈ эмну...,ӄэлёӄы'м эмнуӈкы гым гъурэтигымэ ытԓьатаӈ эмну...,qeɬoqʔəm emnuŋkə ɣəm ɣʔuretiɣəme. ətɬʔataŋ emn...,qeɬoqʔəm emnuŋkə ɣəm ɣʔuretiɣəme ətɬʔataŋ emnu...,33,33,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",[transcription],0,273637
4,birth,Birth,Birth,гым эмнуӈкы тъурэтгъэк гымнинэт ытԓьат гымнин ...,гым эмнуӈкы тъурэтгъэк гымнинэт ытԓьат гымнин ...,ɣəm emnuŋkə tʔuretɣʔek. ɣəmninet ətɬʔat ɣəmnin...,ɣəm emnuŋkə tʔuretɣʔek ɣəmninet ətɬʔat ɣəmnin ...,9,9,"[1, 2, 3, 4, 5, 6, 7, 8, 9]",[transcription],400,81090


## Replacing cyrillic to IPA

In [ ]:
usable_pairs.to_csv('/content/chuklang_full_cyrillic_ipa_pairs.csv', index=False)

In [ ]:
VOWEL_PAIRS_RULES = {
    "э'": 'ʔe',
    "а'": 'ʔa',
    "о'": 'ʔo',
    "и'": 'ʔi',
    "ы'": 'ʔə',
    "у'": 'ʔu',
    'ъя': 'ja',
    'ъе': 'je',
    'ъё': 'jo',
    'ъю': 'ju',
}


IOTATED_VOWEL_RULES = {
    'я': ('a', 'ja'),
    'е': ('e', 'je'),
    'ё': ('o', 'jo'),
    'ю': ('u', 'ju'),
}


CYRILLIC_VOWELS = {
    'а', 'о', 'э', 'е', 'ё', 'и', 'ы', 'у', 'ю', 'я',
}


CYR_TO_CHUKLANG_CHAR_RULES = {
    'ъ': 'ʔ',
    'ь': 'ʔ',

    'ӈ': 'ŋ',
    'ӄ': 'q',
    'ԓ': 'ɬ',
    'Ԓ': 'ɬ',

    'а': 'a',
    'о': 'o',
    'э': 'e',
    'и': 'i',
    'ы': 'ə',
    'у': 'u',

    'т': 't',
    'д': 'd',
    'л': 'ɬ',
    'м': 'm',
    'н': 'n',
    'в': 'w',
    'к': 'k',
    'г': 'ɣ',
    'ч': 's',
    'с': 's',
    'р': 'r',
    'б': 'b',
    'п': 'p',
    'й': 'j',
    'ж': 'ž',
    'з': 'z',
    'ф': 'f',
    'х': 'x',
    'ш': 'š',
}

In [ ]:
def remove_punctuation_keep_apostrophe(text):
    chars = []

    for char in text:
        category = unicodedata.category(char)

        if category.startswith('P') and char != "'":
            chars.append(' ')
        else:
            chars.append(char)

    return ''.join(chars)


def normalize_text_basic(text):
    if pd.isna(text):
        return ''

    text = str(text)
    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    text = text.replace('\u00a0', ' ')
    text = remove_punctuation_keep_apostrophe(text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def transliterate_word_cyrillic_to_chuklang(word):
    result = []
    i = 0

    while i < len(word):
        two_chars = word[i:i + 2]

        if two_chars in VOWEL_PAIRS_RULES:
            result.append(VOWEL_PAIRS_RULES[two_chars])
            i += 2
            continue

        char = word[i]

        if char in IOTATED_VOWEL_RULES:
            plain_value, iotated_value = IOTATED_VOWEL_RULES[char]

            is_word_initial = i == 0
            is_after_vowel = i > 0 and word[i - 1] in CYRILLIC_VOWELS

            if is_word_initial or is_after_vowel:
                result.append(iotated_value)
            else:
                result.append(plain_value)

            i += 1
            continue

        result.append(CYR_TO_CHUKLANG_CHAR_RULES.get(char, char))
        i += 1

    return ''.join(result)


def transliterate_cyrillic_to_chuklang(text):
    text = normalize_text_basic(text)

    words = text.split()

    words = [
        transliterate_word_cyrillic_to_chuklang(word)
        for word in words
    ]

    return ' '.join(words)


def levenshtein_distance(a, b):
    a = str(a)
    b = str(b)

    previous = list(range(len(b) + 1))

    for i, char_a in enumerate(a, start=1):
        current = [i]

        for j, char_b in enumerate(b, start=1):
            insert_cost = current[j - 1] + 1
            delete_cost = previous[j] + 1
            replace_cost = previous[j - 1] + int(char_a != char_b)

            current.append(min(insert_cost, delete_cost, replace_cost))

        previous = current

    return previous[-1]


def cer(reference, hypothesis):
    reference = normalize_text_basic(reference)
    hypothesis = normalize_text_basic(hypothesis)

    if len(reference) == 0:
        return None

    return levenshtein_distance(reference, hypothesis) / len(reference)

In [ ]:
usable_pairs['ipa_predicted'] = usable_pairs['cyrillic_full'].apply(
    transliterate_cyrillic_to_chuklang
)

usable_pairs['ipa_predicted_norm'] = usable_pairs['ipa_predicted'].apply(
    normalize_text_basic
)

## Testing converting results

In [ ]:
def tokenize_words(text):
    text = normalize_text_basic(text)
    return text.split()


def word_edit_ops(reference, hypothesis):
    ref_tokens = tokenize_words(reference)
    hyp_tokens = tokenize_words(hypothesis)

    n = len(ref_tokens)
    m = len(hyp_tokens)

    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0] = i

    for j in range(1, m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            substitute_cost = 0 if ref_tokens[i - 1] == hyp_tokens[j - 1] else 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + substitute_cost
            )

    ops = []
    i = n
    j = m

    while i > 0 or j > 0:
        if (
            i > 0 and
            j > 0 and
            ref_tokens[i - 1] == hyp_tokens[j - 1] and
            dp[i][j] == dp[i - 1][j - 1]
        ):
            ops.append({
                'op': 'equal',
                'ref_token': ref_tokens[i - 1],
                'hyp_token': hyp_tokens[j - 1],
                'ref_pos': i - 1,
                'hyp_pos': j - 1,
            })
            i -= 1
            j -= 1

        elif (
            i > 0 and
            j > 0 and
            dp[i][j] == dp[i - 1][j - 1] + 1
        ):
            ops.append({
                'op': 'replace',
                'ref_token': ref_tokens[i - 1],
                'hyp_token': hyp_tokens[j - 1],
                'ref_pos': i - 1,
                'hyp_pos': j - 1,
            })
            i -= 1
            j -= 1

        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            ops.append({
                'op': 'delete',
                'ref_token': ref_tokens[i - 1],
                'hyp_token': None,
                'ref_pos': i - 1,
                'hyp_pos': j,
            })
            i -= 1

        else:
            ops.append({
                'op': 'insert',
                'ref_token': None,
                'hyp_token': hyp_tokens[j - 1],
                'ref_pos': i,
                'hyp_pos': j - 1,
            })
            j -= 1

    ops.reverse()

    return ops, ref_tokens, hyp_tokens


def make_context(tokens, start, end, context_size=5):
    if not tokens:
        return ''

    start = max(0, start)
    end = min(len(tokens), end)

    left_start = max(0, start - context_size)
    right_end = min(len(tokens), end + context_size)

    left = tokens[left_start:start]
    middle = tokens[start:end]
    right = tokens[end:right_end]

    parts = []

    if left:
        parts.append(' '.join(left))

    if middle:
        parts.append('<<< ' + ' '.join(middle) + ' >>>')
    else:
        parts.append('<<< ∅ >>>')

    if right:
        parts.append(' '.join(right))

    return ' '.join(parts)


def classify_error_type(group_ops):
    types = {op['op'] for op in group_ops}

    if types == {'replace'}:
        return 'substitution'

    if types == {'delete'}:
        return 'deletion'

    if types == {'insert'}:
        return 'insertion'

    return 'mixed'


def extract_error_fragments(reference, hypothesis, cyrillic_text, context_size=5):
    ops, ref_tokens, hyp_tokens = word_edit_ops(reference, hypothesis)

    cyr_tokens = tokenize_words(cyrillic_text)

    error_groups = []
    current_group = []

    for op in ops:
        if op['op'] == 'equal':
            if current_group:
                error_groups.append(current_group)
                current_group = []
        else:
            current_group.append(op)

    if current_group:
        error_groups.append(current_group)

    rows = []

    for group_id, group in enumerate(error_groups, start=1):
        ref_positions = [
            op['ref_pos']
            for op in group
            if op['ref_token'] is not None and op['ref_pos'] is not None
        ]

        hyp_positions = [
            op['hyp_pos']
            for op in group
            if op['hyp_token'] is not None and op['hyp_pos'] is not None
        ]

        ref_fragment_tokens = [
            op['ref_token']
            for op in group
            if op['ref_token'] is not None
        ]

        hyp_fragment_tokens = [
            op['hyp_token']
            for op in group
            if op['hyp_token'] is not None
        ]

        cyr_fragment_tokens = []

        for pos in hyp_positions:
            if 0 <= pos < len(cyr_tokens):
                cyr_fragment_tokens.append(cyr_tokens[pos])

        if ref_positions:
            ref_start = min(ref_positions)
            ref_end = max(ref_positions) + 1
        else:
            ref_start = group[0]['ref_pos']
            ref_end = ref_start

        if hyp_positions:
            hyp_start = min(hyp_positions)
            hyp_end = max(hyp_positions) + 1
        else:
            hyp_start = group[0]['hyp_pos']
            hyp_end = hyp_start

        rows.append({
            'error_group_id': group_id,
            'error_type': classify_error_type(group),

            'cyr_fragment': ' '.join(cyr_fragment_tokens),
            'ref_fragment': ' '.join(ref_fragment_tokens),
            'hyp_fragment': ' '.join(hyp_fragment_tokens),

            'cyr_context': make_context(cyr_tokens, hyp_start, hyp_end, context_size=context_size),
            'ref_context': make_context(ref_tokens, ref_start, ref_end, context_size=context_size),
            'hyp_context': make_context(hyp_tokens, hyp_start, hyp_end, context_size=context_size),

            'n_ref_words': len(ref_fragment_tokens),
            'n_hyp_words': len(hyp_fragment_tokens),
            'n_cyr_words': len(cyr_fragment_tokens),

            'ref_start_word': ref_start,
            'hyp_start_word': hyp_start,
        })

    return pd.DataFrame(rows)

In [ ]:
usable_pairs['transliteration_cer'] = usable_pairs.apply(
    lambda row: cer(row['ipa_full_norm'], row['ipa_predicted_norm']),
    axis=1
)

In [ ]:
all_error_rows = []

for _, row in usable_pairs.iterrows():
    errors = extract_error_fragments(
        reference=row['ipa_full_norm'],
        hypothesis=row['ipa_predicted_norm'],
        cyrillic_text=row['cyrillic_full_norm'],
        context_size=5
    )

    if errors.empty:
        continue

    errors.insert(0, 'recording_key', row['recording_key'])
    errors.insert(1, 'metadata_recording_id', row['metadata_recording_id'])
    errors.insert(2, 'eaf_recording_id', row['eaf_recording_id'])

    all_error_rows.append(errors)

if all_error_rows:
    error_fragments_df = pd.concat(all_error_rows, ignore_index=True)
else:
    error_fragments_df = pd.DataFrame()

In [ ]:
error_fragments_df[
    [
        'cyr_fragment',
        'hyp_fragment',
        'ref_fragment',
    ]
]

,cyr_fragment,hyp_fragment,ref_fragment
0,канчалянэты,kansaɬanetə,kančalanetə
1,вэрталёттэ,wertaɬotte,wertalʲotte
2,абрамович ӈыронвэрталёта,abramowis ŋəronwertaɬota,abramowič ŋəronwertalʲota
3,абрамович,abramowis,abramowič
4,нрзб,nrzb,нрзб
...,...,...,...
248,это рымавыттэ'м,eto rəmawəttʔem,это rəmawətteʔm
249,и,i,и
250,эээ,eee,эээ
251,а,a,а


# Checking the statistical significance of the differences between models

In [4]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.0 MB/s eta 0:00:00


In [8]:
import os
import warnings
from pathlib import Path
import re

import numpy as np
import pandas as pd

import jiwer

In [26]:
REFERENCE_COL = 'reference'
PREDICTION_COL = 'prediction'
PATH_COL = 'path'

BASE_DIR = Path('/content')

MODEL_FILES = {
    'MMS existing ckt': BASE_DIR / 'mms_existing_ckt_chuklang_test_predictions.csv',
    'MMS target-only': BASE_DIR / 'mms_adapter_chuklang_only_chuklang_test_predictions.csv',
    'MMS staged source-eval': BASE_DIR / 'mms_adapter_staged_bible_radio_to_chuklang_source_eval_stage2_chuklang_test_predictions.csv',
    'MMS staged target-eval': BASE_DIR / 'mms_adapter_staged_bible_radio_to_chuklang_stage2_chuklang_test_predictions.csv',
    'MMS pooled': BASE_DIR / 'mms_adapter_pooled_chuklang_test_predictions.csv',

    'XLS-R 1B target-only': BASE_DIR / 'xlsr_1b_chuklang_only_chuklang_test_predictions.csv',
    'XLS-R 1B pooled': BASE_DIR / 'xlsr_1b_pooled_to_chuklang_chuklang_test_predictions.csv',
    'XLS-R 1B staged pooled to target': BASE_DIR / 'xlsr_1b_pooled_to_chuklang_stage2_chuklang_only_chuklang_test_predictions.csv',
    'XLS-R 1B staged source-eval': BASE_DIR / 'xlsr_1b_staged_bible_radio_to_chuklang_source_eval_chuklang_test_predictions.csv',
}

PLANNED_PAIRS = [
    {
        'comparison': 'H1: MMS target-only vs existing ckt',
        'model_a': 'MMS target-only',
        'model_b': 'MMS existing ckt',
    },
    {
        'comparison': 'H2: best MMS vs best XLS-R',
        'model_a': 'MMS staged source-eval',
        'model_b': 'XLS-R 1B staged pooled to target',
    },
    {
        'comparison': 'H3: MMS staged source-eval vs target-only',
        'model_a': 'MMS staged source-eval',
        'model_b': 'MMS target-only',
    },
    {
        'comparison': 'H3: XLS-R staged source-eval vs target-only',
        'model_a': 'XLS-R 1B staged source-eval',
        'model_b': 'XLS-R 1B target-only',
    },
    {
        'comparison': 'H4: MMS staged source-eval vs pooled',
        'model_a': 'MMS staged source-eval',
        'model_b': 'MMS pooled',
    },
    {
        'comparison': 'H4: XLS-R staged source-eval vs pooled',
        'model_a': 'XLS-R 1B staged source-eval',
        'model_b': 'XLS-R 1B pooled',
    },
    {
        'comparison': 'H4: XLS-R staged pooled to target vs pooled',
        'model_a': 'XLS-R 1B staged pooled to target',
        'model_b': 'XLS-R 1B pooled',
    },
    {
        'comparison': 'H5: MMS target-eval vs source-eval',
        'model_a': 'MMS staged target-eval',
        'model_b': 'MMS staged source-eval',
    },

]

In [27]:
def normalize_text_for_metric(text):
    if pd.isna(text):
        return ''

    text = str(text)
    text = ' '.join(text.split())

    return text


def extract_ids_from_path(path_value):
    path_str = str(path_value).replace('\\', '/')
    filename = Path(path_str).name
    stem = Path(filename).stem

    match = re.match(r'^(?P<block_id>.+)_(?P<segment_number>\d+)$', stem)

    if match:
        block_id = match.group('block_id')
    else:
        block_id = stem

    segment_id = stem

    return segment_id, block_id


def load_predictions_file(path):
    df = pd.read_csv(path)

    required_cols = {
        REFERENCE_COL,
        PREDICTION_COL,
        PATH_COL,
    }

    missing_cols = required_cols - set(df.columns)

    if missing_cols:
        raise ValueError(
            f'Missing columns in {path}: {missing_cols}. '
            f'Available columns: {list(df.columns)}'
        )

    ids = df[PATH_COL].apply(extract_ids_from_path)

    result = pd.DataFrame(
        {
            'segment_id': ids.apply(lambda x: x[0]),
            'block_id': ids.apply(lambda x: x[1]),
            'reference': df[REFERENCE_COL].apply(normalize_text_for_metric),
            'hypothesis': df[PREDICTION_COL].apply(normalize_text_for_metric),
        }
    )

    if result['segment_id'].duplicated().any():
        duplicated = result.loc[
            result['segment_id'].duplicated(),
            'segment_id',
        ].head().tolist()

        raise ValueError(
            f'Duplicated segment_id values in {path}. '
            f'Examples: {duplicated}'
        )

    return result


def load_pair_predictions(model_a_name, model_b_name, model_files):
    df_a = load_predictions_file(model_files[model_a_name])
    df_b = load_predictions_file(model_files[model_b_name])

    df_a = df_a.rename(
        columns={
            'hypothesis': 'hyp_a',
            'block_id': 'block_id_a',
            'reference': 'reference_a',
        }
    )

    df_b = df_b.rename(
        columns={
            'hypothesis': 'hyp_b',
            'block_id': 'block_id_b',
            'reference': 'reference_b',
        }
    )

    merged = df_a.merge(df_b, on='segment_id', how='inner')

    if len(merged) != len(df_a) or len(merged) != len(df_b):
        raise ValueError(
            f'Merged size is {len(merged)}, '
            f'but model A has {len(df_a)} rows and model B has {len(df_b)} rows. '
            f'Check segment_id extraction from path.'
        )

    ref_mismatch = (merged['reference_a'] != merged['reference_b']).mean()

    if ref_mismatch > 0:
        raise ValueError(
            f'References differ in {ref_mismatch:.2%} of paired rows. '
            f'This means the files are not safely comparable.'
        )

    block_mismatch = (merged['block_id_a'] != merged['block_id_b']).mean()

    if block_mismatch > 0:
        warnings.warn(
            f'Block IDs differ in {block_mismatch:.2%} of paired rows. '
            f'The block_id from model A will be used.'
        )

    pair_df = pd.DataFrame(
        {
            'segment_id': merged['segment_id'],
            'block_id': merged['block_id_a'],
            'reference': merged['reference_a'],
            'hyp_a': merged['hyp_a'],
            'hyp_b': merged['hyp_b'],
        }
    )

    return pair_df


def get_error_and_denom(reference, hypothesis, metric):
    reference = normalize_text_for_metric(reference)
    hypothesis = normalize_text_for_metric(hypothesis)

    if metric == 'WER':
        output = jiwer.process_words(reference, hypothesis)
    elif metric == 'CER':
        output = jiwer.process_characters(reference, hypothesis)
    else:
        raise ValueError(f'Unknown metric: {metric}')

    errors = (
        output.substitutions
        + output.deletions
        + output.insertions
    )

    denom = (
        output.hits
        + output.substitutions
        + output.deletions
    )

    return errors, denom


def add_metric_stats(pair_df, metric):
    rows = []

    for _, row in pair_df.iterrows():
        err_a, denom_a = get_error_and_denom(
            row['reference'],
            row['hyp_a'],
            metric,
        )

        err_b, denom_b = get_error_and_denom(
            row['reference'],
            row['hyp_b'],
            metric,
        )

        if denom_a != denom_b:
            raise ValueError('Reference denominator mismatch.')

        rows.append(
            {
                'segment_id': row['segment_id'],
                'block_id': row['block_id'],
                'err_a': err_a,
                'err_b': err_b,
                'denom': denom_a,
            }
        )

    stats_df = pd.DataFrame(rows)

    if (stats_df['denom'] == 0).any():
        warnings.warn('Some segments have empty references and will be excluded.')
        stats_df = stats_df[stats_df['denom'] > 0].copy()

    return stats_df


def aggregate_metric(stats_df):
    total_denom = stats_df['denom'].sum()

    value_a = stats_df['err_a'].sum() / total_denom
    value_b = stats_df['err_b'].sum() / total_denom
    delta = value_a - value_b

    return value_a, value_b, delta


def paired_blockwise_bootstrap(
    stats_df,
    n_bootstrap=10000,
    confidence=0.95,
    seed=42,
):
    rng = np.random.default_rng(seed)

    block_stats = (
        stats_df
        .groupby('block_id', as_index=False)
        [['err_a', 'err_b', 'denom']]
        .sum()
    )

    n_blocks = len(block_stats)

    if n_blocks < 2:
        raise ValueError(
            f'Only {n_blocks} block found. '
            f'Blockwise bootstrap requires at least 2 blocks.'
        )

    deltas = np.empty(n_bootstrap)

    for i in range(n_bootstrap):
        sampled_idx = rng.integers(0, n_blocks, size=n_blocks)
        sample = block_stats.iloc[sampled_idx]

        value_a = sample['err_a'].sum() / sample['denom'].sum()
        value_b = sample['err_b'].sum() / sample['denom'].sum()

        deltas[i] = value_a - value_b

    alpha = 1 - confidence
    ci_low = np.quantile(deltas, alpha / 2)
    ci_high = np.quantile(deltas, 1 - alpha / 2)

    significant = ci_high < 0 or ci_low > 0

    return {
        'ci_low': ci_low,
        'ci_high': ci_high,
        'significant_95ci': significant,
        'n_blocks': n_blocks,
    }


def compare_models_with_bootstrap(
    comparison_name,
    model_a,
    model_b,
    model_files,
    metrics=('WER', 'CER'),
    n_bootstrap=10000,
    seed=42,
):
    pair_df = load_pair_predictions(
        model_a,
        model_b,
        model_files,
    )

    results = []

    for metric in metrics:
        stats_df = add_metric_stats(pair_df, metric)

        value_a, value_b, delta = aggregate_metric(stats_df)

        bootstrap_result = paired_blockwise_bootstrap(
            stats_df,
            n_bootstrap=n_bootstrap,
            seed=seed,
        )

        results.append(
            {
                'comparison': comparison_name,
                'metric': metric,
                'model_a': model_a,
                'model_b': model_b,
                'value_a': value_a,
                'value_b': value_b,
                'delta_a_minus_b': delta,
                'ci_low': bootstrap_result['ci_low'],
                'ci_high': bootstrap_result['ci_high'],
                'significant_95ci': bootstrap_result['significant_95ci'],
                'n_blocks': bootstrap_result['n_blocks'],
                'n_segments': len(pair_df),
            }
        )

    return results

In [28]:
all_results = []

for pair in PLANNED_PAIRS:
    pair_results = compare_models_with_bootstrap(
        comparison_name=pair['comparison'],
        model_a=pair['model_a'],
        model_b=pair['model_b'],
        model_files=MODEL_FILES,
        metrics=('WER', 'CER'),
        n_bootstrap=10000,
        seed=42,
    )

    all_results.extend(pair_results)

significance_df = pd.DataFrame(all_results)

for col in ['value_a', 'value_b', 'delta_a_minus_b', 'ci_low', 'ci_high']:
    significance_df[col] = significance_df[col].round(4)

significance_df

,comparison,metric,model_a,model_b,value_a,value_b,delta_a_minus_b,ci_low,ci_high,significant_95ci,n_blocks,n_segments
0,H1: MMS target-only vs existing ckt,WER,MMS target-only,MMS existing ckt,0.7685,0.9236,-0.1551,-0.1982,-0.1130,True,58,200
1,H1: MMS target-only vs existing ckt,CER,MMS target-only,MMS existing ckt,0.1954,0.2999,-0.1045,-0.1250,-0.0859,True,58,200
2,H2: best MMS vs best XLS-R,WER,MMS staged source-eval,XLS-R 1B staged pooled to target,0.7391,0.7779,-0.0388,-0.0839,0.0083,False,58,200
3,H2: best MMS vs best XLS-R,CER,MMS staged source-eval,XLS-R 1B staged pooled to target,0.1825,0.2054,-0.0229,-0.0341,-0.0111,True,58,200
4,H3: MMS staged source-eval vs target-only,WER,MMS staged source-eval,MMS target-only,0.7391,0.7685,-0.0294,-0.0561,0.0000,False,58,200
5,H3: MMS staged source-eval vs target-only,CER,MMS staged source-eval,MMS target-only,0.1825,0.1954,-0.0128,-0.0216,-0.0045,True,58,200
6,H3: XLS-R staged source-eval vs target-only,WER,XLS-R 1B staged source-eval,XLS-R 1B target-only,0.8860,0.9706,-0.0846,-0.1210,-0.0519,True,58,200
7,H3: XLS-R staged source-eval vs target-only,CER,XLS-R 1B staged source-eval,XLS-R 1B target-only,0.2750,0.3333,-0.0583,-0.0713,-0.0451,True,58,200
8,H4: MMS staged source-eval vs pooled,WER,MMS staged source-eval,MMS pooled,0.7391,0.9013,-0.1622,-0.2050,-0.1232,True,58,200
9,H4: MMS staged source-eval vs pooled,CER,MMS staged source-eval,MMS pooled,0.1825,0.2558,-0.0733,-0.0865,-0.0601,True,58,200


In [29]:
significance_df.to_csv(
    '/content/chuklang_test_planned_significance_comparisons.csv',
    index=False,
)